<div style="background:linear-gradient(120deg,#01172F 0%,#082744 55%,#14476B 100%);
            border-radius:14px;padding:34px 38px;color:#EAEFF5;
            font-family:system-ui,-apple-system,'Segoe UI',sans-serif;">
  <div style="display:inline-block;border:1px solid #D9A441;color:#D9A441;
              border-radius:999px;padding:3px 13px;font-size:12px;
              letter-spacing:1.6px;font-weight:700;">PROJETO FINAL &nbsp;·&nbsp; VISÃO COMPUTACIONAL</div>
  <h1 style="margin:16px 0 6px 0;font-size:34px;line-height:1.15;color:#FFFFFF;">
     Detecção e <span style="color:#D9A441;">laudo automático</span> de tumor cerebral
  </h1>
  <p style="margin:0;font-size:16px;color:#A9BCD0;max-width:820px;">
     Um pipeline completo com <b style="color:#EAEFF5;">Ultralytics YOLO26-L</b>:
     partição por protocolo de aquisição, síntese de lesões, treino com orçamento
     de tempo fechado, diagnóstico quantitativo do modelo e uma interface web funcional.
  </p>
  <div style="margin-top:22px;display:flex;gap:26px;flex-wrap:wrap;font-size:13px;color:#A9BCD0;">
    <div><span style="color:#D9A441;font-weight:700;">Dataset</span><br>brain-tumor (Ultralytics)</div>
    <div><span style="color:#D9A441;font-weight:700;">Modelo</span><br>YOLO26-L · end-to-end, sem NMS</div>
    <div><span style="color:#D9A441;font-weight:700;">Orçamento</span><br>≤ 30 min de execução</div>
    <div><span style="color:#D9A441;font-weight:700;">Saída</span><br>laudo estruturado + app web</div>
  </div>
</div>

---

### O que este notebook faz

O notebook anterior mostrou o YOLO **usando** um modelo pronto. Aqui a pergunta é outra:
**como se constrói um sistema inteiro em torno de um detector — e como se prova que ele funciona?**

A resposta tem sete movimentos, e cada seção abaixo é um deles:

| # | Movimento | Pergunta que ele responde |
|---|---|---|
| 1 | Entender o YOLO26 | O que mudou na arquitetura e o que isso muda no meu código? |
| 2 | Conhecer o dado | O que exatamente está anotado nessas imagens? |
| 3 | Particionar com honestidade | Meu conjunto de teste está mesmo *fora* do treino? |
| 4 | Sintetizar dados | Como fabrico exemplos novos sem inventar rótulo errado? |
| 5 | Treinar sob orçamento | Como garanto que isso roda em 30 minutos em qualquer GPU? |
| 6 | Diagnosticar o modelo | Onde e por que ele erra — e com qual confiança devo acreditar nele? |
| 7 | Entregar | Como transformo tensores em um laudo que uma pessoa lê? |

> **Não há mágica. Há matemática!** 🧙 — e, neste projeto, também há **contabilidade**:
> cada minuto de execução é medido e prestado conta no final.

---

> ### ⚠️ Aviso obrigatório
> Este é um **exercício acadêmico de visão computacional**. O sistema aqui construído
> **não é um dispositivo médico**, não passou por validação clínica e **não deve ser
> usado para qualquer decisão diagnóstica**. O rótulo `positive`/`negative` do dataset
> é uma anotação de conveniência do conjunto público, não um diagnóstico verificado.

## 0 · Antes de tudo

**Acesse o menu: `Ambiente de execução` → `Alterar o tipo de ambiente de execução` →
defina o acelerador de hardware como `GPU` (T4 já basta).**

Sem GPU o notebook ainda roda, mas o treino não cabe no orçamento de 30 minutos —
o código detecta isso sozinho e avisa.

In [ ]:
# Bibliotecas. O `-q` mantém a saída limpa; o `-U` garante uma versão do
# Ultralytics que já conhece a família YOLO26.
!pip install -q -U ultralytics
!pip install -q flask

### 0.1 · Configuração central

Tudo o que muda o comportamento do projeto mora **em um único objeto**. Isso não é
preciosismo: é o que permite, no fim do notebook, rastrear cada número de volta até
os parâmetros que o produziram.

Três perfis vêm prontos:

| perfil | modelo | orçamento de treino | quando usar |
|---|---|---|---|
| `rapido`   | YOLO26-S | 6 min  | primeira passada, aula ao vivo |
| `padrao`   | **YOLO26-L** | **15 min** | **entrega deste projeto** |
| `completo` | YOLO26-X | 45 min | quando o relógio não importa |

O perfil `padrao` é o que respeita o teto de 30 minutos de ponta a ponta.

In [ ]:
import os, sys, time, json, glob, math, shutil, random, collections
import numpy as np, pandas as pd, torch, cv2
import matplotlib.pyplot as plt

# ---- ambiente ------------------------------------------------------------
EM_COLAB = os.path.isdir("/content")
RAIZ     = "/content/projeto" if EM_COLAB else os.path.join(os.getcwd(), "projeto")
DEVICE   = 0 if torch.cuda.is_available() else "cpu"
os.makedirs(RAIZ, exist_ok=True); os.chdir(RAIZ)
os.makedirs("fig", exist_ok=True)

print(f"Colab .......: {EM_COLAB}")
print(f"Raiz ........: {RAIZ}")
print(f"PyTorch .....: {torch.__version__}")
print(f"GPU .........: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'INDISPONÍVEL'}")
if not torch.cuda.is_available():
    print("\n⚠️  Sem GPU o treino do YOLO26-L não cabe em 30 min.")
    print("   Troque o ambiente de execução ou use MODO = 'rapido'.")

O objeto `Config` abaixo é a única fonte de verdade do projeto. Repare no campo
`orcamento_treino_min`: ele não é uma estimativa, é uma **restrição** — a seção 5
explica como ela é imposta ao treinador.

In [ ]:
import os, time, json, glob, shutil, sys
from dataclasses import dataclass, field, asdict
import numpy as np

# --------------------------------------------------------------- config
@dataclass
class Config:
    modelo: str = "yolo26l.pt"        # requisito do projeto: L no mínimo
    imgsz: int = 640
    lote: int = 16                    # "auto" também é aceito pelo Ultralytics
    orcamento_total_min: float = 30.0
    orcamento_treino_min: float = 15.0   # teto rígido passado ao treinador
    epocas_teto: int = 120            # só um teto; `time` decide o real
    sementes: int = 42
    n_sinteticas: int = 300
    usar_sinteticas: bool = True
    recall_minimo_triagem: float = 0.85
    iou_consolidacao: float = 0.55
    conf_coleta: float = 0.05
    artefatos_amostra: int = 90       # imagens usadas no teste de robustez
    severidades: tuple = (1, 2, 3)
    device: int | str = 0
    raiz: str = "/content/projeto"

    def dicionario(self):
        d = asdict(self); d["severidades"] = list(d["severidades"]); return d

MODOS = {
    # perfis prontos; o orçamento de treino é o que realmente controla o relógio
    "rapido":   dict(modelo="yolo26s.pt", imgsz=512, orcamento_treino_min=6.0,
                     n_sinteticas=150, artefatos_amostra=60),
    "padrao":   dict(modelo="yolo26l.pt", imgsz=640, orcamento_treino_min=15.0,
                     n_sinteticas=300, artefatos_amostra=90),
    "completo": dict(modelo="yolo26x.pt", imgsz=640, orcamento_treino_min=45.0,
                     n_sinteticas=600, artefatos_amostra=223),
    # perfil de verificação: roda o notebook inteiro em CPU só para provar que
    # nenhuma célula quebra. Não produz resultado com significado científico.
    "teste":    dict(modelo="yolo26n.pt", imgsz=320, orcamento_treino_min=1.0,
                     n_sinteticas=20, artefatos_amostra=8, lote=4,
                     epocas_teto=3, severidades=(1, 3)),
}

def configurar(modo="padrao", **ajustes):
    c = Config(**MODOS[modo]); [setattr(c, k, v) for k, v in ajustes.items()]
    return c

MODO = os.environ.get("MODO_PROJETO", "padrao")     # 'rapido' | 'padrao' | 'completo'
cfg  = configurar(MODO, device=DEVICE, raiz=RAIZ)
print(f"Perfil ativo: {MODO}")
for k, v in cfg.dicionario().items():
    print(f"  {k:.<28} {v}")

### 0.2 · O cronômetro

Um projeto que promete rodar em 30 minutos precisa **medir** os 30 minutos. A classe
abaixo marca cada etapa, mostra quanto sobra do orçamento e, no fim, desenha para onde
o tempo foi. É o mesmo raciocínio de um orçamento de obra: o valor prometido só vale
alguma coisa se houver prestação de contas.

In [ ]:
# ------------------------------------------------------------ cronômetro
class Cronometro:
    """Mede cada etapa e confronta o total com o orçamento declarado."""
    def __init__(self, orcamento_min=30.0):
        self.t0 = time.time(); self.orcamento = orcamento_min * 60
        self.etapas = []; self._ultimo = self.t0

    def marco(self, nome, silencioso=False):
        agora = time.time()
        dt = agora - self._ultimo; self._ultimo = agora
        self.etapas.append(dict(etapa=nome, segundos=dt,
                                acumulado=agora - self.t0))
        if not silencioso:
            restante = self.orcamento - (agora - self.t0)
            print(f"⏱  {nome:<38} {dt:6.1f}s   "
                  f"acumulado {(agora-self.t0)/60:5.1f} min   "
                  f"restante {restante/60:5.1f} min")

    def restante_min(self):
        return (self.orcamento - (time.time() - self.t0)) / 60

    def tabela(self):
        import pandas as pd
        df = pd.DataFrame(self.etapas)
        df["minutos"] = df["segundos"] / 60
        df["% do orçamento"] = 100 * df["segundos"] / self.orcamento
        return df[["etapa", "minutos", "% do orçamento"]].round(2)

    def figura(self, caminho=None):
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(9.5, max(3, 0.42 * len(self.etapas))))
        nomes = [e["etapa"] for e in self.etapas]
        mins = [e["segundos"] / 60 for e in self.etapas]
        inicio = np.concatenate([[0], np.cumsum(mins)[:-1]])
        cores = [PALETA["roxo"] if "reino" in n else PALETA["primaria"] for n in nomes]
        ax.barh(nomes[::-1], mins[::-1], left=inicio[::-1], color=cores[::-1])
        ax.axvline(self.orcamento / 60, color=PALETA["magenta"], ls="--", lw=1.8)
        ax.text(self.orcamento / 60, -0.6, f" orçamento {self.orcamento/60:.0f} min",
                color=PALETA["magenta"], fontsize=9, fontweight="bold")
        ax.set_xlabel("minutos desde o início")
        titular(ax, f"Onde foram os {sum(mins):.1f} minutos",
                "barra âmbar = treino; a linha vermelha é o teto declarado")
        fig.tight_layout(); moldura(fig)
        if caminho:
            os.makedirs(os.path.dirname(caminho) or ".", exist_ok=True)
            fig.savefig(caminho); return caminho
        return fig

crono = Cronometro(orcamento_min=cfg.orcamento_total_min)
crono.marco("setup do ambiente")

### 0.3 · Identidade visual

Todo o projeto — figuras, relatório e interface web — usa **a paleta do template Dtox**,
lida diretamente de `scss/_variables.scss`: azul `#008DEC`, tinta escura `#091337`, texto
`#4D546F`, cinza de seção `#F2F3F5` e os gradientes `#17FFD3 → #D3FC71` e
`#17FFD3 → #23E3EE`. As cores de apoio (roxo, magenta, laranja, verde) vêm das formas
decorativas do próprio template.

Não é enfeite: quando dezoito figuras e quatro páginas web usam a mesma codificação de
cor, o leitor para de decodificar legenda e passa a ler o conteúdo. A fonte também é a
do template, **Poppins** — a célula abaixo a instala se o ambiente não a tiver.

In [ ]:
# Todas as cores abaixo vêm do próprio template:
#   scss/_variables.scss  ->  $primary-color, $text-color, $text-color-dark,
#                             $gray, $primary-gradient, $secondary-gradient
#   images/background-shape/*.png -> cores das formas decorativas
# Nada aqui foi inventado: é o sistema visual do template aplicado às figuras.
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

PALETA = {
    # _variables.scss
    "primaria":  "#008DEC",   # $primary-color
    "escuro":    "#091337",   # $text-color-dark  (títulos)
    "texto":     "#4D546F",   # $text-color       (corpo)
    "cinza":     "#F2F3F5",   # $gray             (fundos de seção)
    "branco":    "#FFFFFF",   # $body-color
    # $primary-gradient / $secondary-gradient
    "turquesa":  "#17FFD3",
    "lima":      "#D3FC71",
    "ciano":     "#23E3EE",
    # formas decorativas do template
    "azul_forte":"#0070F0",
    "roxo":      "#9000F0",
    "verde":     "#10F030",
    "magenta":   "#F04090",
    "laranja":   "#F09000",
    "borda":     "#E3E7EE",
}
# ciclo de cores das séries, na ordem em que o template usa suas cores
CICLO = [PALETA["primaria"], PALETA["turquesa"], PALETA["roxo"], PALETA["laranja"],
         PALETA["magenta"], PALETA["verde"], PALETA["azul_forte"], PALETA["lima"]]

# rampa contínua reproduzindo o $secondary-gradient do template
CMAP_DTOX = LinearSegmentedColormap.from_list(
    "dtox", [PALETA["escuro"], "#0B3B7A", PALETA["primaria"],
             PALETA["ciano"], PALETA["turquesa"], PALETA["lima"]])

# a fonte do template é Poppins; se não estiver instalada, cai em DejaVu Sans
def _familia_disponivel():
    try:
        from matplotlib import font_manager
        nomes = {f.name for f in font_manager.fontManager.ttflist}
        for c in ("Poppins", "Montserrat", "DejaVu Sans"):
            if c in nomes:
                return c
    except Exception:
        pass
    return "DejaVu Sans"

FONTE = _familia_disponivel()

def aplicar_estilo():
    mpl.rcParams.update({
        "figure.facecolor": PALETA["branco"],
        "axes.facecolor":   PALETA["branco"],
        "savefig.facecolor":PALETA["branco"],
        "axes.edgecolor":   PALETA["borda"],
        "axes.labelcolor":  PALETA["texto"],
        "axes.titlecolor":  PALETA["escuro"],
        "axes.titleweight": "bold",
        "axes.titlesize":   13,
        "axes.labelsize":   10,
        "axes.grid":        True,
        "axes.axisbelow":   True,
        "grid.color":       PALETA["cinza"],
        "grid.linewidth":   1.0,
        "text.color":       PALETA["texto"],
        "xtick.color":      PALETA["texto"],
        "ytick.color":      PALETA["texto"],
        "xtick.labelsize":  9, "ytick.labelsize": 9,
        "legend.frameon":   True,
        "legend.framealpha":1.0,
        "legend.edgecolor": PALETA["borda"],
        "legend.fancybox":  True,
        "font.family":      FONTE,
        "font.weight":      "normal",
        "figure.dpi":       110,
        "savefig.dpi":      160,
        "savefig.bbox":     "tight",
        "axes.prop_cycle":  mpl.cycler(color=CICLO),
        "lines.linewidth":  2.4,
        "patch.linewidth":  0,
    })

def garantir_poppins(pasta="/tmp/fontes"):
    """
    Tenta instalar a Poppins (a fonte do template) para o matplotlib.
    Se o download falhar — sem rede, espelho fora do ar —, o projeto segue com a
    fonte de fallback e as figuras continuam corretas: é um enfeite, não um requisito.
    Devolve o nome da família que ficou ativa.
    """
    global FONTE
    from matplotlib import font_manager
    if "Poppins" in {f.name for f in font_manager.fontManager.ttflist}:
        FONTE = "Poppins"; aplicar_estilo(); return FONTE
    import os, urllib.request
    base = ("https://github.com/google/fonts/raw/main/ofl/poppins/Poppins-%s.ttf")
    os.makedirs(pasta, exist_ok=True)
    baixadas = 0
    for peso in ("Light", "Regular", "Medium", "SemiBold", "Bold"):
        destino = os.path.join(pasta, f"Poppins-{peso}.ttf")
        try:
            if not os.path.exists(destino):
                urllib.request.urlretrieve(base % peso, destino)
            font_manager.fontManager.addfont(destino)
            baixadas += 1
        except Exception:
            pass
    FONTE = "Poppins" if baixadas else _familia_disponivel()
    aplicar_estilo()
    return FONTE

def titular(ax, titulo, subtitulo=None):
    """Título no padrão do template: h2 escuro + parágrafo de apoio."""
    ax.set_title(titulo, loc="left", pad=16 if subtitulo else 9)
    if subtitulo:
        ax.text(0, 1.015, subtitulo, transform=ax.transAxes,
                fontsize=8.5, color=PALETA["texto"], va="bottom", alpha=.85)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(PALETA["borda"])
    ax.spines["bottom"].set_color(PALETA["primaria"])
    ax.spines["bottom"].set_linewidth(1.8)
    return ax

def moldura(fig, rotulo="Projeto Final · YOLO26 · Detecção e Laudo de Tumor Cerebral"):
    fig.text(0.005, -0.02, rotulo, fontsize=7.5, color=PALETA["texto"], ha="left", alpha=.7)
    return fig

aplicar_estilo()
ativa = garantir_poppins()          # a fonte do template; cai em outra se não vier
print("Fonte das figuras:", ativa)
print("Paleta (lida de scss/_variables.scss do Dtox):")
for k, v in PALETA.items():
    print(f"  {k:<11} {v}")

### 0.4 · A biblioteca de figuras

Dezoito figuras saem deste notebook. Escrever cada uma no lugar onde ela aparece
produziria centenas de linhas de matplotlib espalhadas pelo texto e afogaria a
explicação. Toda a plotagem mora aqui, uma função por figura, e as seções seguintes
chamam apenas o nome.

In [ ]:
import os, math, collections
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import sys

COR_CLASSE = {0: PALETA["primaria"], 1: PALETA["roxo"]}
COR_STATUS = {"TP": PALETA["verde"], "FP": PALETA["magenta"], "FN": PALETA["laranja"]}

def salvar(fig, caminho):
    os.makedirs(os.path.dirname(caminho) or ".", exist_ok=True)
    moldura(fig); fig.savefig(caminho); return caminho

# ---------------------------------------------------------- 1. EDA
def fig_composicao_particoes(resumo, caminho=None):
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    nomes = [r["particao"] for r in resumo]
    x = np.arange(len(nomes)); larg = 0.38
    ax[0].bar(x - larg/2, [r["negative"] for r in resumo], larg,
              label="negative", color=PALETA["primaria"])
    ax[0].bar(x + larg/2, [r["positive"] for r in resumo], larg,
              label="positive", color=PALETA["roxo"])
    ax[0].set_xticks(x); ax[0].set_xticklabels(nomes); ax[0].legend()
    titular(ax[0], "Caixas por classe", "desequilíbrio muda entre partições")
    ax[1].bar(x, [r["imagens"] for r in resumo], 0.55, color=PALETA["primaria"])
    for i, r in enumerate(resumo):
        ax[1].text(i, r["imagens"], str(r["imagens"]), ha="center", va="bottom", fontsize=9)
    ax[1].set_xticks(x); ax[1].set_xticklabels(nomes)
    titular(ax[1], "Imagens por partição")
    prop = [100 * r["positive"] / max(r["negative"] + r["positive"], 1) for r in resumo]
    ax[2].bar(x, prop, 0.55, color=[PALETA["primaria"], PALETA["primaria"], PALETA["magenta"]][:len(x)])
    ax[2].axhline(prop[0], ls="--", lw=1.4, color=PALETA["texto"])
    for i, v in enumerate(prop):
        ax[2].text(i, v, f"{v:.0f}%", ha="center", va="bottom", fontsize=9)
    ax[2].set_xticks(x); ax[2].set_xticklabels(nomes); ax[2].set_ylabel("% positive")
    titular(ax[2], "Prevalência da classe positive", "a linha tracejada é o treino")
    fig.tight_layout()
    return salvar(fig, caminho) if caminho else fig

def fig_geometria_e_area(registros_por_particao, caminho=None):
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    # geometrias
    todas = collections.Counter()
    for regs in registros_por_particao.values():
        for r in regs:
            todas[(r["largura"], r["altura"])] += 1
    itens = todas.most_common(8)
    rot = [f"{w}×{h}" for (w, h), _ in itens]
    ax[0].barh(rot[::-1], [v for _, v in itens][::-1], color=PALETA["primaria"])
    titular(ax[0], "Protocolos de aquisição", "geometria da imagem como proxy de fonte")
    # área relativa das caixas
    for i, (nome, regs) in enumerate(registros_por_particao.items()):
        areas = [c[3] * c[4] * 100 for r in regs for c in r["caixas"]]
        if areas:
            ax[1].hist(areas, bins=np.logspace(-1, 1.2, 30), histtype="step", lw=2,
                       label=nome, color=CICLO[i])
    ax[1].set_xscale("log"); ax[1].set_xlabel("área da caixa (% da imagem)")
    ax[1].legend(); titular(ax[1], "Escala dos alvos", "mediana ~1,5% → objeto pequeno")
    # nº de caixas por imagem
    for i, (nome, regs) in enumerate(registros_por_particao.items()):
        c = collections.Counter(r["n_caixas"] for r in regs)
        xs = sorted(c); ax[2].plot(xs, [c[k] for k in xs], "o-", label=nome, color=CICLO[i])
    ax[2].set_xlabel("caixas por imagem"); ax[2].set_yscale("symlog"); ax[2].legend()
    titular(ax[2], "Densidade de anotação")
    fig.tight_layout()
    return salvar(fig, caminho) if caminho else fig

def fig_mapa_calor_centros(registros_por_particao, caminho=None, n=64):
    fig, ax = plt.subplots(1, len(registros_por_particao),
                           figsize=(4.4 * len(registros_por_particao), 4.2))
    ax = np.atleast_1d(ax)
    for i, (nome, regs) in enumerate(registros_por_particao.items()):
        H = np.zeros((n, n))
        for r in regs:
            for c in r["caixas"]:
                xi = min(n - 1, int(c[1] * n)); yi = min(n - 1, int(c[2] * n))
                H[yi, xi] += 1
        H = np.clip(H, 0, np.percentile(H[H > 0], 99) if (H > 0).any() else 1)
        im = ax[i].imshow(H, cmap=CMAP_DTOX, origin="upper")
        ax[i].set_xticks([]); ax[i].set_yticks([])
        ax[i].set_title(f"{nome}", color=PALETA["escuro"], fontweight="bold")
        fig.colorbar(im, ax=ax[i], fraction=0.046, pad=0.03)
    fig.suptitle("Onde as lesões aparecem no campo de visão",
                 color=PALETA["escuro"], fontweight="bold")
    fig.tight_layout()
    return salvar(fig, caminho) if caminho else fig

# ------------------------------------------------ 2. curvas de treino
def fig_curvas_treino(csv_resultados, caminho=None):
    import pandas as pd
    df = pd.read_csv(csv_resultados)
    df.columns = [c.strip() for c in df.columns]
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    perdas = [c for c in df.columns if c.startswith("train/")]
    for i, c in enumerate(perdas):
        ax[0].plot(df["epoch"], df[c], label=c.split("/")[-1], color=CICLO[i])
    ax[0].set_xlabel("época"); ax[0].legend(fontsize=8)
    titular(ax[0], "Perdas de treino")
    vperdas = [c for c in df.columns if c.startswith("val/")]
    for i, c in enumerate(vperdas):
        ax[1].plot(df["epoch"], df[c], label=c.split("/")[-1], color=CICLO[i])
    ax[1].set_xlabel("época"); ax[1].legend(fontsize=8)
    titular(ax[1], "Perdas de validação", "divergência aqui = sobreajuste")
    for i, c in enumerate([c for c in df.columns if c.startswith("metrics/")]):
        ax[2].plot(df["epoch"], df[c], label=c.replace("metrics/", "").replace("(B)", ""),
                   color=CICLO[i])
    ax[2].set_xlabel("época"); ax[2].legend(fontsize=8); ax[2].set_ylim(0, 1)
    titular(ax[2], "Métricas por época")
    fig.tight_layout()
    return salvar(fig, caminho) if caminho else fig

# ------------------------------------------------ 3. PR / F1 / limiar
def fig_pr_e_limiar(curvas, varredura, ponto, nomes, caminho=None):
    fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
    for c, dados in curvas.items():
        if len(dados["recall"]) > 1:
            ax[0].plot(dados["recall"], dados["precision"], color=COR_CLASSE.get(c, CICLO[c]),
                       label=f"{nomes[c]} · AP@50={dados['ap']:.3f}")
    ax[0].set_xlabel("recall"); ax[0].set_ylabel("precisão")
    ax[0].set_xlim(0, 1); ax[0].set_ylim(0, 1.02); ax[0].legend(fontsize=8)
    titular(ax[0], "Curva precisão–recall", "área sob a curva = AP")
    t = [l["limiar"] for l in varredura]
    ax[1].plot(t, [l["precisao"] for l in varredura], label="precisão", color=PALETA["primaria"])
    ax[1].plot(t, [l["recall"] for l in varredura], label="recall", color=PALETA["ciano"])
    ax[1].plot(t, [l["f1"] for l in varredura], label="F1", color=PALETA["roxo"], lw=2.8)
    if ponto["f1"]:
        ax[1].axvline(ponto["f1"]["limiar"], ls="--", color=PALETA["escuro"], lw=1.3)
        ax[1].annotate(f"τ*={ponto['f1']['limiar']:.2f}",
                       (ponto["f1"]["limiar"], ponto["f1"]["f1"]),
                       textcoords="offset points", xytext=(6, 8), fontsize=9,
                       color=PALETA["escuro"], fontweight="bold")
    if ponto.get("triagem"):
        ax[1].axvline(ponto["triagem"]["limiar"], ls=":", color=PALETA["magenta"], lw=1.6)
        ax[1].annotate(f"τ_triagem={ponto['triagem']['limiar']:.2f}",
                       (ponto["triagem"]["limiar"], 0.05), textcoords="offset points",
                       xytext=(6, 0), fontsize=8.5, color=PALETA["magenta"])
    ax[1].set_xlabel("limiar de confiança τ"); ax[1].legend(fontsize=8); ax[1].set_ylim(0, 1.02)
    titular(ax[1], "Escolha do ponto de operação", "não existe τ universal: existe τ para um objetivo")
    ax[2].plot([l["FP"] for l in varredura], [l["TP"] for l in varredura],
               color=PALETA["primaria"])
    ax[2].set_xlabel("falsos positivos acumulados"); ax[2].set_ylabel("verdadeiros positivos")
    titular(ax[2], "Compromisso TP × FP", "o eixo que o radiologista sente")
    fig.tight_layout()
    return salvar(fig, caminho) if caminho else fig

# ------------------------------------------------ 4. matriz de confusão
def fig_matriz_confusao(M, rotulos, caminho=None, titulo="Matriz de confusão"):
    Mn = M / np.maximum(M.sum(axis=0, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(5.6, 5))
    im = ax.imshow(Mn, cmap=CMAP_DTOX, vmin=0, vmax=1)
    ax.set_xticks(range(len(rotulos))); ax.set_xticklabels(rotulos, rotation=20)
    ax.set_yticks(range(len(rotulos))); ax.set_yticklabels(rotulos)
    ax.set_xlabel("verdadeiro"); ax.set_ylabel("predito")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i,j]}\n{Mn[i,j]*100:.0f}%", ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color=PALETA["branco"] if Mn[i, j] < 0.55 else PALETA["escuro"])
    ax.grid(False); ax.set_title(titulo, color=PALETA["escuro"], fontweight="bold", loc="left")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    return salvar(fig, caminho) if caminho else fig

# ------------------------------------------------ 5. calibração
def fig_calibracao(linhas, valor_ece, caminho=None):
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.2))
    c = [l["centro"] for l in linhas]
    acc = [np.nan if l["n"] == 0 else l["acuracia"] for l in linhas]
    ax[0].plot([0, 1], [0, 1], ls="--", color=PALETA["texto"], label="calibração perfeita")
    ax[0].bar(c, acc, width=0.09, color=PALETA["primaria"], edgecolor=PALETA["escuro"],
              linewidth=0.8, label="acurácia observada")
    ax[0].set_xlabel("confiança predita"); ax[0].set_ylabel("fração de acertos")
    ax[0].set_xlim(0, 1); ax[0].set_ylim(0, 1); ax[0].legend(fontsize=8)
    titular(ax[0], f"Diagrama de confiabilidade  ·  ECE = {valor_ece:.3f}",
            "acima da diagonal = subconfiante; abaixo = superconfiante")
    ax[1].bar(c, [l["n"] for l in linhas], width=0.09, color=PALETA["ciano"])
    ax[1].set_xlabel("confiança predita"); ax[1].set_ylabel("nº de detecções")
    titular(ax[1], "Massa por faixa de confiança")
    fig.tight_layout()
    return salvar(fig, caminho) if caminho else fig

# ------------------------------------------------ 6. taxonomia de erros
def fig_taxonomia(tax, caminho=None):
    rot = {"classificacao": "classe errada (IoU ok)",
           "localizacao": "caixa mal posicionada",
           "cls+loc": "classe e caixa erradas",
           "duplicata": "duplicata",
           "fundo": "invenção sobre o fundo",
           "nao_detectado": "lesão não detectada"}
    itens = [(rot[k], v) for k, v in tax["erros"].items()]
    itens.sort(key=lambda x: x[1])
    fig, ax = plt.subplots(figsize=(8.2, 4))
    cores = [PALETA["magenta"] if "não detectada" in n else PALETA["primaria"]
             for n, _ in itens]
    ax.barh([n for n, _ in itens], [v for _, v in itens], color=cores)
    for i, (_, v) in enumerate(itens):
        ax.text(v, i, f" {v}", va="center", fontsize=9, color=PALETA["escuro"])
    titular(ax, f"Anatomia do erro  ·  {tax['tp']} acertos",
            "onde o modelo falha importa mais do que quanto ele falha")
    fig.tight_layout()
    return salvar(fig, caminho) if caminho else fig

# ------------------------------------------------ 7. robustez
def fig_robustez(tabela, caminho=None):
    """tabela: {artefato: {severidade: map50}} com severidade 0 = original."""
    fig, ax = plt.subplots(figsize=(8.6, 4.4))
    for i, (nome, serie) in enumerate(tabela.items()):
        sev = sorted(serie)
        ax.plot(sev, [serie[s] for s in sev], "o-", label=nome, color=CICLO[i])
    ax.set_xlabel("severidade do artefato"); ax.set_ylabel("mAP@50")
    ax.set_xticks(sorted({s for v in tabela.values() for s in v}))
    ax.legend(fontsize=8.5)
    titular(ax, "Degradação sob artefatos de ressonância",
            "severidade 0 = imagem original; queda acentuada = fragilidade")
    fig.tight_layout()
    return salvar(fig, caminho) if caminho else fig

# ------------------------------------------------ 8. comparativo de domínios
def fig_dominios(resultados, caminho=None):
    """resultados: lista de dict(nome, map50, map50_95, recall, precisao)."""
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
    nomes = [r["nome"] for r in resultados]; x = np.arange(len(nomes)); w = 0.38
    ax[0].bar(x - w/2, [r["map50"] for r in resultados], w, label="mAP@50", color=PALETA["primaria"])
    ax[0].bar(x + w/2, [r["map50_95"] for r in resultados], w, label="mAP@50-95", color=PALETA["turquesa"])
    for i, r in enumerate(resultados):
        ax[0].text(i - w/2, r["map50"], f"{r['map50']:.3f}", ha="center", va="bottom", fontsize=8)
        ax[0].text(i + w/2, r["map50_95"], f"{r['map50_95']:.3f}", ha="center", va="bottom", fontsize=8)
    ax[0].set_xticks(x); ax[0].set_xticklabels(nomes, fontsize=9); ax[0].legend(fontsize=8)
    ax[0].set_ylim(0, 1)
    titular(ax[0], "Generalização entre domínios", "queda interno→externo = custo do deslocamento")
    ax[1].bar(x - w/2, [r.get("precisao", 0) for r in resultados], w, label="precisão", color=PALETA["primaria"])
    ax[1].bar(x + w/2, [r.get("recall", 0) for r in resultados], w, label="recall", color=PALETA["ciano"])
    ax[1].set_xticks(x); ax[1].set_xticklabels(nomes, fontsize=9); ax[1].legend(fontsize=8)
    ax[1].set_ylim(0, 1)
    titular(ax[1], "Precisão e recall no ponto de operação")
    fig.tight_layout()
    return salvar(fig, caminho) if caminho else fig

# ------------------------------------------------ 9. grade qualitativa
def fig_qualitativa(registros, nomes, limiar=0.25, n=6, caminho=None, semente=0):
    import cv2
    rng = np.random.default_rng(semente)
    alvo = [r for r in registros if r["gt_cx"]]
    idx = rng.choice(len(alvo), size=min(n, len(alvo)), replace=False)
    cols = 3; linhas = math.ceil(len(idx) / cols)
    fig, ax = plt.subplots(linhas, cols, figsize=(4.2 * cols, 4.2 * linhas))
    ax = np.atleast_1d(ax).ravel()
    for k, i in enumerate(idx):
        r = alvo[i]
        img = cv2.imread(r["caminho"], cv2.IMREAD_GRAYSCALE)
        ax[k].imshow(img, cmap="gray"); ax[k].axis("off")
        # recorta o enquadramento no encéfalo para não desperdiçar área com fundo
        try:
            ys, xs = np.where(mascara_encefalo(img) > 0)
            if len(xs):
                mg = 0.06 * max(img.shape)
                ax[k].set_xlim(max(0, xs.min()-mg), min(img.shape[1], xs.max()+mg))
                ax[k].set_ylim(min(img.shape[0], ys.max()+mg), max(0, ys.min()-mg))
        except Exception:
            pass
        keep = [j for j, c in enumerate(r["pr_cf"]) if c >= limiar]
        res, falt = casar_imagem(r["gt_cx"], r["gt_cls"],
                                 [r["pr_cx"][j] for j in keep],
                                 [r["pr_cf"][j] for j in keep],
                                 [r["pr_cls"][j] for j in keep])
        for (j, tipo, cat, _, _) in res:
            x1, y1, x2, y2 = r["pr_cx"][keep[j]]
            cor = COR_STATUS["TP"] if tipo == "TP" else COR_STATUS["FP"]
            ax[k].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, ec=cor, lw=2))
            ax[k].text(x1, y1 - 3, f"{nomes[r['pr_cls'][keep[j]]]} {r['pr_cf'][keep[j]]:.2f}",
                       color=cor, fontsize=7.5, fontweight="bold")
        for j in falt:
            x1, y1, x2, y2 = r["gt_cx"][j]
            ax[k].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                                      ec=COR_STATUS["FN"], lw=2, ls="--"))
            ax[k].text(x1, y2 + 12, "não detectado", color=COR_STATUS["FN"],
                       fontsize=7.5, fontweight="bold")
        ax[k].set_title(os.path.basename(r["caminho"]), fontsize=8, color=PALETA["escuro"])
    for a in ax[len(idx):]:
        a.axis("off")
    fig.suptitle("Verde = acerto · Magenta = falso positivo · Laranja tracejado = lesão perdida",
                 color=PALETA["escuro"], fontweight="bold", fontsize=11)
    fig.tight_layout()
    return salvar(fig, caminho) if caminho else fig

---
## 1 · O que exatamente mudou no YOLO26

Antes de treinar, vale entender o que se está treinando — porque três decisões de
arquitetura do YOLO26 **mudam o código que escrevemos depois**.

**1. Cabeça end-to-end, sem NMS.**
As versões anteriores emitiam centenas de caixas candidatas e um pós-processamento
(*Non-Maximum Suppression*) escolhia as sobreviventes. O YOLO26 treina uma cabeça
*one-to-one*: idealmente uma caixa por objeto, já na saída da rede. Consequência
prática: os botões `iou=` e `agnostic_nms=` deixam de ter o efeito de antes, e
`augment=True` (test-time augmentation) **não é suportado** — vamos precisar de outra
estratégia para estimar incerteza (seção 7).

**2. Sem Distribution Focal Loss (DFL).**
O DFL modelava cada coordenada como uma distribuição discreta sobre *bins*. Removê-lo
simplifica a cabeça e mantém a regressão sem faixa limitada — o que ajuda justamente
em alvos pequenos, que é o nosso caso.

**3. STAL e ProgLoss.**
*Small-Target-Aware Label Assignment* garante que alvos pequenos continuem recebendo
âncoras positivas durante a atribuição de rótulos; a *Progressive Loss* desloca o peso
do treino ao longo das épocas em direção à arquitetura que de fato roda na inferência.
Ambos importam aqui: a mediana das nossas lesões ocupa cerca de **1,5% da área da
imagem** — território clássico de *small object detection*.

> Fonte: documentação oficial do YOLO26 (`docs.ultralytics.com/models/yolo26`).
> Os nomes e o comportamento acima vêm de lá; os efeitos práticos em `augment=` e
> `iou=` foram **verificados experimentalmente** durante a construção deste notebook.

In [ ]:
from ultralytics import YOLO
import ultralytics
print("Ultralytics:", ultralytics.__version__)

modelo_coco = YOLO(cfg.modelo)          # pesos pré-treinados em COCO (80 classes)
n_par = sum(p.numel() for p in modelo_coco.model.parameters())
print(f"Modelo ........: {cfg.modelo}")
print(f"Parâmetros ....: {n_par/1e6:.2f} M")
print(f"Tarefa ........: {modelo_coco.task}")
print(f"Classes COCO ..: {len(modelo_coco.names)} (person, bicycle, car, ...)")
crono.marco("carga do modelo pré-treinado")

Com o modelo carregado e a arquitetura entendida, falta o que realmente decide o
resultado de qualquer projeto de visão: **o dado**.

---
## 2 · O dataset primário: `brain-tumor` da Ultralytics

O conjunto traz cortes de ressonância magnética do encéfalo com caixas anotadas em
duas classes, `negative` e `positive`. Ele é distribuído pela própria Ultralytics,
baixa sem credencial e pesa 4,3 MB — três propriedades que fazem diferença quando o
projeto precisa ser reproduzível por outra pessoa em outra máquina.

**Uma honestidade necessária sobre os rótulos.** Neste conjunto público, `negative` e
`positive` são *classes de caixa*, não um diagnóstico verificado por laudo clínico —
praticamente toda imagem anotada tem uma caixa. Portanto o que estamos aprendendo é
"localizar a região de interesse e classificá-la na convenção do dataset", **não**
"diagnosticar tumor". Essa distinção reaparece na seção 9 (limitações) e é o motivo
do aviso no topo do notebook.

In [ ]:
!mkdir -p datasets && cd datasets &&   wget -q -O brain-tumor.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/brain-tumor.zip &&   unzip -q -o brain-tumor.zip -d brain-tumor && rm -f brain-tumor.zip

DS = os.path.join(RAIZ, "datasets", "brain-tumor")
print("Conteúdo:", sorted(os.listdir(DS)))
for s in ("train", "val"):
    print(f"  images/{s}: {len(os.listdir(f'{DS}/images/{s}')):4d}   "
          f"labels/{s}: {len(os.listdir(f'{DS}/labels/{s}')):4d}")
crono.marco("download do dataset primário")

### 2.1 · Inventário: ler o dado antes de treinar nele

A função abaixo não usa nenhum atalho de biblioteca: abre cada imagem, lê cada arquivo
de rótulo e devolve uma lista de registros. Parece trabalho desnecessário — é o oposto.
É exatamente essa leitura crua que revela, daqui a duas células, que o dataset esconde
**vários protocolos de aquisição diferentes** misturados no mesmo split.

In [ ]:
import os, glob, shutil, json, collections
from PIL import Image

GEOMETRIA_EXTERNA = (192, 256)      # protocolo mantido 100% fora do treino

def ler_rotulos(caminho_txt):
    if not os.path.exists(caminho_txt):
        return []
    linhas = [l.strip() for l in open(caminho_txt).read().strip().split("\n") if l.strip()]
    saida = []
    for l in linhas:
        p = l.split()
        saida.append((int(p[0]), *[float(v) for v in p[1:5]]))
    return saida

def caminho_rotulo(caminho_img):
    return os.path.splitext(caminho_img.replace("/images/", "/labels/"))[0] + ".txt"

def inventariar(raiz_ds):
    """Lê todo o dataset original e devolve uma lista de registros."""
    reg = []
    for split in ("train", "val"):
        for ip in sorted(glob.glob(f"{raiz_ds}/images/{split}/*")):
            with Image.open(ip) as im:
                w, h = im.size; modo = im.mode
            cxs = ler_rotulos(caminho_rotulo(ip))
            reg.append(dict(caminho=ip, split_original=split, largura=w, altura=h,
                            modo=modo, n_caixas=len(cxs), caixas=cxs))
    return reg

registros = inventariar(DS)
print(f"{len(registros)} imagens inventariadas")
df_inv = pd.DataFrame([{k: v for k, v in r.items() if k != "caixas"} for r in registros])
display(df_inv.head())
print("\nProtocolos de aquisição encontrados (largura×altura):")
display(df_inv.groupby(["largura", "altura"]).size()
        .sort_values(ascending=False).rename("imagens").to_frame().head(10))
crono.marco("inventário do dataset")

### 2.2 · O que a tabela acima revela

Não é um dataset homogêneo. Há pelo menos **três geometrias dominantes** — 512×512,
192×256 e 256×256 — e mais um punhado de tamanhos raros. Em imagem médica, geometria
diferente costuma significar **equipamento, protocolo ou fonte diferente**: outra
relação sinal-ruído, outro campo de visão, outra estatística de intensidade.

E aqui está o detalhe que decide o desenho de todo o experimento:

- as **175 imagens 192×256 estão todas no split de treino** original;
- o split de validação original **não contém nenhuma delas**.

Ou seja: validar no split original mede o modelo apenas em protocolos que ele já viu.
É uma medida legítima, mas incompleta — e a seção 3 conserta isso.

---
## 3 · Particionar com honestidade: o *domain shift* que já estava no dataset

Um split aleatório mede a capacidade de generalizar **para mais do mesmo**. É a
pergunta fácil. A pergunta difícil — e a única que importa quando o sistema sai do
notebook — é: *e quando chegar uma imagem de outro aparelho?*

Em vez de importar um segundo dataset de fora (o que traria dependência de credencial
e minutos de download), este projeto extrai o segundo conjunto **do próprio dado**,
usando a geometria de aquisição como *proxy* de fonte:

| partição | o que é | papel |
|---|---|---|
| `treino`  | protocolos vistos, split de treino | ajuste dos pesos |
| `interno` | protocolos vistos, split de validação | validação *in-distribution* |
| `externo` | **todo** o protocolo 192×256 | validação sob *domain shift* puro |

A regra é dura de propósito: **nenhuma** imagem 192×256 entra no treino. Não é um
holdout aleatório, é um **holdout de protocolo** — a versão honesta da pergunta.

In [ ]:
def particionar(reg, geometria_externa=GEOMETRIA_EXTERNA):
    """
    D_treino  : protocolos vistos, split train original
    D_interno : protocolos vistos, split val original  (validação in-distribution)
    D_externo : TODAS as imagens do protocolo `geometria_externa` (domain shift puro)
    """
    d = {"treino": [], "interno": [], "externo": []}
    for r in reg:
        if (r["largura"], r["altura"]) == geometria_externa:
            d["externo"].append(r)
        elif r["split_original"] == "train":
            d["treino"].append(r)
        else:
            d["interno"].append(r)
    return d

def materializar(particoes, destino, nomes=("negative", "positive")):
    """Copia arquivos para uma árvore YOLO e escreve os data.yaml."""
    os.makedirs(destino, exist_ok=True)
    for nome, regs in particoes.items():
        for sub in ("images", "labels"):
            os.makedirs(f"{destino}/{sub}/{nome}", exist_ok=True)
        for r in regs:
            base = os.path.basename(r["caminho"])
            stem = os.path.splitext(base)[0]
            shutil.copy(r["caminho"], f"{destino}/images/{nome}/{base}")
            lp = caminho_rotulo(r["caminho"])
            alvo = f"{destino}/labels/{nome}/{stem}.txt"
            if os.path.exists(lp):
                shutil.copy(lp, alvo)
            else:
                open(alvo, "w").close()          # imagem de fundo, rótulo vazio
    yamls = {}
    bloco_nomes = "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(nomes))
    for val_nome in ("interno", "externo"):
        p = f"{destino}/data_{val_nome}.yaml"
        open(p, "w").write(
            f"# gerado automaticamente pelo pipeline\npath: {os.path.abspath(destino)}\n"
            f"train: images/treino\nval: images/{val_nome}\n\n{bloco_nomes}")
        yamls[val_nome] = p
    return yamls

def resumo(particoes):
    linhas = []
    for nome, regs in particoes.items():
        cls = collections.Counter()
        geo = collections.Counter()
        vazias = 0
        for r in regs:
            geo[(r["largura"], r["altura"])] += 1
            if r["n_caixas"] == 0: vazias += 1
            for c in r["caixas"]: cls[c[0]] += 1
        linhas.append(dict(particao=nome, imagens=len(regs), caixas=sum(cls.values()),
                           negative=cls[0], positive=cls[1], sem_caixa=vazias,
                           geometrias=len(geo),
                           geometria_dominante=max(geo, key=geo.get) if geo else None))
    return linhas

particoes = particionar(registros)
BASE = os.path.join(RAIZ, "particionado")
yamls = materializar(particoes, BASE)
tab_part = pd.DataFrame(resumo(particoes))
display(tab_part)
print("\nYAMLs de validação:", yamls)
crono.marco("particionamento por protocolo")

Leia a tabela com atenção, porque ela contém o achado mais interessante do projeto:

- no **treino**, `negative` e `positive` estão quase equilibradas (≈46% positivas);
- no conjunto **externo**, **84% das caixas são `positive`**.

Isso não é só deslocamento de covariável (a imagem mudou); é **deslocamento de rótulo**
(a prevalência mudou). São dois problemas diferentes, e a seção 6.4 vai mostrar que
eles cobram preços diferentes.

In [ ]:
caminho = fig_composicao_particoes(resumo(particoes), "fig/01_composicao.png")
plt.show()
caminho = fig_geometria_e_area(particoes, "fig/02_geometria.png")
plt.show()
caminho = fig_mapa_calor_centros(particoes, "fig/03_mapa_calor.png")
plt.show()
crono.marco("análise exploratória (3 figuras)")

### 3.1 · Controle negativo: um conjunto onde a resposta certa é "nada"

Falta ainda um terceiro tipo de teste, que quase nenhum projeto de aula faz e que é
justamente o que separa um detector útil de um gerador de alarme falso:

> **O que o modelo faz diante de uma imagem médica que não é uma ressonância de crânio?**

A resposta certa é *nenhum achado*. Para medir isso usamos o `medical-pills`, também
da Ultralytics: 115 imagens médicas de outro domínio inteiro. Toda caixa que o modelo
emitir ali é, por construção, **invenção** — e a taxa dessas invenções é uma métrica
de segurança tão importante quanto o mAP.

In [ ]:
!cd datasets && \
  wget -q -O medical-pills.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/medical-pills.zip && \
  unzip -q -o medical-pills.zip -d medical-pills && rm -f medical-pills.zip

OOD = sorted(glob.glob(os.path.join(RAIZ, "datasets/medical-pills/images/*/*")))
print(f"Controle negativo (OOD): {len(OOD)} imagens de outro domínio médico")
crono.marco("download do controle negativo")

### 3.2 · Prova de que o modelo pronto não resolve o problema

Agora que conhecemos o dado, cabe a demonstração que fecha a seção 1: apontar o
detector **pré-treinado em COCO** para uma ressonância e ver o que acontece.

É o momento em que *transfer learning* deixa de ser jargão. Os pesos do COCO não
servem como **classificador** — não existe a classe "lesão" lá dentro. Mas as camadas
iniciais já aprenderam borda, textura e contraste, e é sobre esse alicerce que o
ajuste fino da seção 5 vai construir.

In [ ]:
amostra = sorted(glob.glob(f"{BASE}/images/interno/*"))[:3]
r_coco = modelo_coco.predict(amostra, imgsz=cfg.imgsz, conf=0.10,
                             device=cfg.device, verbose=False)

fig, ax = plt.subplots(1, 3, figsize=(13, 4.6))
for a, res, cam in zip(ax, r_coco, amostra):
    a.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)); a.axis("off")
    achados = [modelo_coco.names[int(c)] for c in res.boxes.cls] if len(res.boxes) else []
    a.set_title(f"{os.path.basename(cam)}\nCOCO diz: {achados or 'nada'}",
                fontsize=9, color=PALETA["escuro"])
fig.suptitle("YOLO26-L pré-treinado em COCO, sem ajuste fino",
             color=PALETA["escuro"], fontweight="bold")
plt.tight_layout(); plt.show()
crono.marco("baseline zero-shot COCO")

---
## 4 · Dados sintéticos: fabricar exemplos sem fabricar mentira

Temos 718 imagens de treino. Para um detector de 26 milhões de parâmetros, isso é
pouco. A saída óbvia — aumento de dados — o Ultralytics já faz sozinho (mosaico,
HSV, escala, espelhamento). O que ele **não** faz é criar *lesões novas em posições
novas*, e é exatamente isso que mais falta aqui.

Este projeto usa duas famílias de síntese, com propósitos opostos:

| família | o que produz | para quê |
|---|---|---|
| **4.1 · copy-paste de lesões** | imagens novas **com rótulo exato** | entram no **treino** |
| **4.2 · artefatos de RM** | versões degradadas de imagens conhecidas | entram só no **teste de robustez** |

A separação é deliberada. Misturar as duas coisas seria treinar e testar na mesma
distribuição artificial — o erro metodológico mais comum em trabalhos com dado sintético.

### 4.1 · A matemática do copy-paste

Colar um recorte por cima de outra imagem produz uma costura visível, e o detector
aprende a costura em vez da lesão. Quatro cuidados evitam isso:

**(a) Onde colar.** Só dentro do encéfalo. A máscara sai de Otsu seguido do maior
componente conexo e de um fechamento morfológico $\;M = (B \oplus K) \ominus K$,
com preenchimento de contorno. Depois uma erosão afasta a colagem da calota craniana.

**(b) Como misturar.** Máscara elíptica com borda difusa: para o pixel a distância
normalizada $d$ do centro,
$$\alpha(x,y)=\mathrm{clip}\!\left(\frac{1-d}{s},\,0,\,1\right),\qquad
  I_{\text{saída}} = \alpha\,I_{\text{lesão}} + (1-\alpha)\,I_{\text{fundo}}$$

**(c) Harmonização de intensidade.** O recorte vem de outra imagem, com outro brilho.
Casamos os dois primeiros momentos com a vizinhança do destino:
$$I' = \frac{I-\mu_{\text{lesão}}}{\sigma_{\text{lesão}}}\,\sigma_{\text{vizinhança}} + \mu_{\text{vizinhança}}$$

**(d) Alternativa por Poisson.** Em parte das amostras usamos *seamless cloning*, que
resolve $\nabla^2 f = \nabla^2 g$ dentro da região com $f|_{\partial\Omega}=I_{\text{fundo}}|_{\partial\Omega}$:
a costura some porque a solução impõe continuidade no contorno.

E o ponto mais importante: **o rótulo não é estimado, é conhecido**. Sabemos onde
colamos, logo a caixa é exata por construção. Dado sintético só vale quando o rótulo
vem de graça junto com o pixel.

In [ ]:
import os, glob, math, random
import numpy as np
import cv2

# ---------------------------------------------------------------- utilidades
def carregar_cinza(caminho):
    img = cv2.imread(caminho, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(caminho)
    return img

def yolo_para_pixel(cx, cy, bw, bh, W, H):
    x1 = (cx - bw / 2) * W; y1 = (cy - bh / 2) * H
    x2 = (cx + bw / 2) * W; y2 = (cy + bh / 2) * H
    return int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))

def pixel_para_yolo(x1, y1, x2, y2, W, H):
    return ((x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H)

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

In [ ]:
# ------------------------------------------------- segmentação do encéfalo
def mascara_encefalo(img, fechamento=9):
    """
    Máscara grosseira da região intracraniana: Otsu -> maior componente conexo
    -> fechamento morfológico -> preenchimento de buracos.
    Serve apenas para restringir ONDE uma lesão sintética pode ser colada.
    """
    borrada = cv2.GaussianBlur(img, (5, 5), 0)
    _, bin_ = cv2.threshold(borrada, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (fechamento, fechamento))
    bin_ = cv2.morphologyEx(bin_, cv2.MORPH_CLOSE, k)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(bin_, 8)
    if n <= 1:
        return np.ones_like(img, np.uint8) * 255
    maior = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    m = np.where(lab == maior, 255, 0).astype(np.uint8)
    contornos, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cheia = np.zeros_like(m)
    cv2.drawContours(cheia, contornos, -1, 255, cv2.FILLED)
    return cheia

In [ ]:
# ------------------------------------------------------- banco de lesões
def banco_de_lesoes(registros, margem=0.15, lado_min=16):
    """Recorta cada caixa anotada; guarda o recorte e a classe de origem."""
    banco = []
    for r in registros:
        if not r["caixas"]:
            continue
        img = carregar_cinza(r["caminho"])
        H, W = img.shape
        for (c, cx, cy, bw, bh) in r["caixas"]:
            x1, y1, x2, y2 = yolo_para_pixel(cx, cy, bw, bh, W, H)
            mx, my = int((x2-x1) * margem), int((y2-y1) * margem)
            x1, y1 = max(0, x1-mx), max(0, y1-my)
            x2, y2 = min(W, x2+mx), min(H, y2+my)
            if x2-x1 < lado_min or y2-y1 < lado_min:
                continue
            banco.append(dict(classe=c, recorte=img[y1:y2, x1:x2].copy(),
                              origem=os.path.basename(r["caminho"])))
    return banco

def _mascara_suave(h, w, suavidade=0.30):
    """Máscara elíptica com borda difusa (feathering) — evita costura visível."""
    yy, xx = np.mgrid[0:h, 0:w]
    cy, cx = (h - 1) / 2, (w - 1) / 2
    d = np.sqrt(((xx - cx) / (w / 2)) ** 2 + ((yy - cy) / (h / 2)) ** 2)
    m = np.clip((1.0 - d) / max(suavidade, 1e-6), 0, 1)
    return cv2.GaussianBlur(m.astype(np.float32), (0, 0), sigmaX=max(h, w) * 0.05)

def _casar_intensidade(recorte, vizinhanca, peso=0.7):
    """Casa média/desvio do recorte com a vizinhança do destino (harmonização)."""
    a = recorte.astype(np.float32)
    mu_a, sd_a = a.mean(), a.std() + 1e-6
    mu_b, sd_b = float(vizinhanca.mean()), float(vizinhanca.std()) + 1e-6
    ajustado = (a - mu_a) / sd_a * sd_b + mu_b
    return np.clip(peso * ajustado + (1 - peso) * a, 0, 255)

def colar_lesao(img, caixas_existentes, banco, rng, escala=(0.7, 1.3),
                tentativas=40, modo="alfa"):
    """
    Insere uma lesão do banco numa posição plausível (dentro do encéfalo,
    sem sobrepor caixas já existentes). Devolve (img_nova, caixa_nova) ou None.
    """
    H, W = img.shape
    masc = mascara_encefalo(img)
    # erosão para não colar rente à calota craniana
    er = cv2.erode(masc, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21)))
    ys, xs = np.where(er > 0)
    if len(xs) == 0:
        return None
    for _ in range(tentativas):
        item = banco[rng.randrange(len(banco))]
        rec = item["recorte"]
        s = rng.uniform(*escala)
        h = max(12, int(rec.shape[0] * s)); w = max(12, int(rec.shape[1] * s))
        if h >= H * 0.6 or w >= W * 0.6:
            continue
        rec_r = cv2.resize(rec, (w, h), interpolation=cv2.INTER_LINEAR)
        if rng.random() < 0.5:
            rec_r = cv2.flip(rec_r, 1)
        i = rng.randrange(len(xs))
        cx, cy = int(xs[i]), int(ys[i])
        x1, y1 = cx - w // 2, cy - h // 2
        x2, y2 = x1 + w, y1 + h
        if x1 < 0 or y1 < 0 or x2 > W or y2 > H:
            continue
        # a lesão precisa cair majoritariamente dentro do encéfalo
        if er[y1:y2, x1:x2].mean() < 200:
            continue
        nova = (x1, y1, x2, y2)
        if any(iou(nova, c) > 0.02 for c in caixas_existentes):
            continue
        destino = img.copy()
        viz = img[max(0, y1-8):min(H, y2+8), max(0, x1-8):min(W, x2+8)]
        rec_h = _casar_intensidade(rec_r, viz)
        if modo == "poisson":
            src = cv2.cvtColor(rec_h.astype(np.uint8), cv2.COLOR_GRAY2BGR)
            dst = cv2.cvtColor(destino, cv2.COLOR_GRAY2BGR)
            centro = (int((x1 + x2) / 2), int((y1 + y2) / 2))
            mk = (_mascara_suave(h, w) > 0.35).astype(np.uint8) * 255
            fundido = cv2.seamlessClone(src, dst, mk, centro, cv2.MIXED_CLONE)
            destino = cv2.cvtColor(fundido, cv2.COLOR_BGR2GRAY)
        else:
            alfa = _mascara_suave(h, w)[..., None][:, :, 0]
            regiao = destino[y1:y2, x1:x2].astype(np.float32)
            destino[y1:y2, x1:x2] = np.clip(alfa * rec_h + (1 - alfa) * regiao,
                                            0, 255).astype(np.uint8)
        return destino, (item["classe"], *nova)
    return None

def gerar_conjunto_sintetico(registros_treino, destino_img, destino_lab, banco,
                             n_alvo=300, semente=42, prob_poisson=0.35,
                             max_lesoes=2, prefixo="sint"):
    """Gera n_alvo imagens sintéticas com rótulos YOLO exatos."""
    rng = random.Random(semente)
    os.makedirs(destino_img, exist_ok=True); os.makedirs(destino_lab, exist_ok=True)
    base = [r for r in registros_treino]
    produzidas, tentativas, i = [], 0, 0
    while len(produzidas) < n_alvo and tentativas < n_alvo * 6:
        tentativas += 1
        r = base[rng.randrange(len(base))]
        img = carregar_cinza(r["caminho"])
        H, W = img.shape
        caixas_px = [yolo_para_pixel(cx, cy, bw, bh, W, H) for (_, cx, cy, bw, bh) in r["caixas"]]
        classes = [c for (c, *_ ) in r["caixas"]]
        novas = []
        k = rng.randint(1, max_lesoes)
        atual = img
        for _ in range(k):
            modo = "poisson" if rng.random() < prob_poisson else "alfa"
            saida = colar_lesao(atual, caixas_px + [n[1:] for n in novas], banco, rng, modo=modo)
            if saida is None:
                break
            atual, cx_nova = saida
            novas.append(cx_nova)
            caixas_px.append(cx_nova[1:])
        if not novas:
            continue
        i += 1
        nome = f"{prefixo}_{i:04d}"
        cv2.imwrite(f"{destino_img}/{nome}.jpg", atual, [cv2.IMWRITE_JPEG_QUALITY, 95])
        linhas = []
        for c, (cx, cy, bw, bh) in zip(classes, [pixel_para_yolo(*b, W, H) for b in caixas_px[:len(classes)]]):
            linhas.append(f"{c} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        for (c, x1, y1, x2, y2) in novas:
            cx, cy, bw, bh = pixel_para_yolo(x1, y1, x2, y2, W, H)
            linhas.append(f"{c} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        open(f"{destino_lab}/{nome}.txt", "w").write("\n".join(linhas) + "\n")
        produzidas.append(dict(nome=nome, origem=os.path.basename(r["caminho"]),
                               n_lesoes_novas=len(novas), n_caixas=len(linhas)))
    return produzidas

In [ ]:
SINT_IMG = f"{BASE}/images/sintetico"
SINT_LAB = f"{BASE}/labels/sintetico"
banco = banco_de_lesoes(particoes["treino"])
print(f"Banco de lesões: {len(banco)} recortes extraídos do treino")

sinteticas = gerar_conjunto_sintetico(
    particoes["treino"], SINT_IMG, SINT_LAB, banco,
    n_alvo=cfg.n_sinteticas if cfg.usar_sinteticas else 0, semente=cfg.sementes)
print(f"Imagens sintéticas geradas: {len(sinteticas)}")
display(pd.DataFrame(sinteticas).head())
crono.marco("síntese de lesões (copy-paste)")

Inspeção visual obrigatória. Dado sintético que ninguém olhou é dado sintético que
ninguém confere — e um artefato de colagem mal resolvido vira atalho para o modelo.

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(14, 7.4))
for j, p in enumerate(sinteticas[:4]):
    orig = carregar_cinza(os.path.join(BASE, "images/treino", p["origem"]))
    sin  = carregar_cinza(f"{SINT_IMG}/{p['nome']}.jpg")
    H, W = sin.shape
    ax[0, j].imshow(orig, cmap="gray"); ax[0, j].set_title(f"original · {p['origem']}", fontsize=8.5)
    ax[1, j].imshow(sin,  cmap="gray"); ax[1, j].set_title(f"sintética · {p['nome']}", fontsize=8.5)
    for l in open(f"{SINT_LAB}/{p['nome']}.txt"):
        c, cx, cy, bw, bh = l.split()
        x1, y1, x2, y2 = yolo_para_pixel(float(cx), float(cy), float(bw), float(bh), W, H)
        ax[1, j].add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, lw=1.9,
                           ec=PALETA["roxo"] if int(c) == 1 else PALETA["primaria"]))
    for a in (ax[0, j], ax[1, j]): a.axis("off")
fig.suptitle("Lesões sintéticas: rótulo exato porque a posição é escolhida, não estimada",
             color=PALETA["escuro"], fontweight="bold")
plt.tight_layout(); plt.savefig("fig/04_sinteticas.png", dpi=150); plt.show()

### 4.2 · Artefatos de ressonância: física simulada, não ruído genérico

Adicionar ruído gaussiano a uma imagem de RM é fisicamente errado. A imagem de
magnitude vem do módulo de um sinal complexo, e cada canal carrega ruído gaussiano
independente — o resultado segue uma **distribuição Riciana**:

$$I_{\text{obs}} = \sqrt{(I + n_1)^2 + n_2^2}, \qquad n_1, n_2 \sim \mathcal{N}(0,\sigma^2)$$

que só se aproxima de uma gaussiana quando a relação sinal-ruído é alta. Nos quatro
artefatos abaixo, três nascem diretamente do espaço-k:

| artefato | origem física | como simulamos |
|---|---|---|
| **ruído Riciano** | ruído térmico no sinal complexo | fórmula acima |
| **campo de bias** | não-uniformidade de $B_1$ | multiplicação por campo polinomial suave |
| **ghosting N/2** | instabilidade entre linhas de codificação de fase | fase $e^{i\pi\lambda}$ em linhas alternadas do espaço-k |
| **ringing de Gibbs** | truncamento do espaço-k | zerar as altas frequências e voltar por IFFT |

Nenhum deles entra no treino. Todos entram no **teste de robustez** da seção 6.5 —
que responde: *quanto de desempenho o modelo perde quando a aquisição sai do ideal?*

In [ ]:
# =====================================================================
# Artefatos de ressonância magnética (apenas para o teste de robustez)
# =====================================================================
def ruido_riciano(img, sigma):
    """Magnitude de ruído gaussiano complexo: |(I+n1) + i·n2| — modelo correto de RM."""
    rng = np.random.default_rng(0)
    a = img.astype(np.float32)
    n1 = rng.normal(0, sigma, a.shape); n2 = rng.normal(0, sigma, a.shape)
    return np.clip(np.sqrt((a + n1) ** 2 + n2 ** 2), 0, 255).astype(np.uint8)

def campo_de_bias(img, amplitude):
    """Não-uniformidade multiplicativa suave (inomogeneidade de B1)."""
    H, W = img.shape
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
    xx = xx / W - 0.5; yy = yy / H - 0.5
    campo = 1.0 + amplitude * (0.9 * xx + 0.6 * yy + 1.4 * (xx ** 2 - yy ** 2) - 0.5)
    return np.clip(img.astype(np.float32) * campo, 0, 255).astype(np.uint8)

def ghosting(img, intensidade, periodo=2):
    """Ghosting N/2: modulação de fase em linhas alternadas do espaço-k."""
    F = np.fft.fftshift(np.fft.fft2(img.astype(np.float32)))
    fase = np.ones(F.shape, dtype=np.complex64)
    fase[::periodo, :] = np.exp(1j * np.pi * intensidade)
    return np.clip(np.abs(np.fft.ifft2(np.fft.ifftshift(F * fase))), 0, 255).astype(np.uint8)

def gibbs(img, fracao_mantida):
    """Truncamento do espaço-k -> ringing de Gibbs nas bordas de alto contraste."""
    F = np.fft.fftshift(np.fft.fft2(img.astype(np.float32)))
    H, W = img.shape
    mh, mw = int(H * fracao_mantida / 2), int(W * fracao_mantida / 2)
    masc = np.zeros_like(F, dtype=np.float32)
    masc[H//2-mh:H//2+mh, W//2-mw:W//2+mw] = 1.0
    return np.clip(np.abs(np.fft.ifft2(np.fft.ifftshift(F * masc))), 0, 255).astype(np.uint8)

# severidade 1..3 por artefato (parâmetros calibrados visualmente)
ARTEFATOS = {
    "ruido_riciano": (ruido_riciano, {1: 8,    2: 16,   3: 28}),
    "campo_de_bias": (campo_de_bias, {1: 0.35, 2: 0.65, 3: 1.00}),
    "ghosting":      (ghosting,      {1: 0.25, 2: 0.50, 3: 0.85}),
    "gibbs":         (gibbs,         {1: 0.40, 2: 0.24, 3: 0.14}),
}

def aplicar_artefato(img, nome, severidade):
    fn, tabela = ARTEFATOS[nome]
    return fn(img, tabela[severidade])

In [ ]:
img_demo = carregar_cinza(sorted(glob.glob(f"{BASE}/images/interno/*"))[3])
fig, ax = plt.subplots(len(ARTEFATOS), 4, figsize=(12.5, 3.1 * len(ARTEFATOS)))
for i, nome in enumerate(ARTEFATOS):
    ax[i, 0].imshow(img_demo, cmap="gray")
    ax[i, 0].set_ylabel(nome.replace("_", " "), fontsize=9.5, color=PALETA["escuro"])
    if i == 0: ax[i, 0].set_title("original", fontsize=9.5)
    for s in cfg.severidades:
        ax[i, s].imshow(aplicar_artefato(img_demo, nome, s), cmap="gray")
        if i == 0: ax[i, s].set_title(f"severidade {s}", fontsize=9.5)
    for a in ax[i]: a.set_xticks([]); a.set_yticks([])
fig.suptitle("Degradações de aquisição simuladas a partir do modelo físico",
             color=PALETA["escuro"], fontweight="bold")
plt.tight_layout(); plt.savefig("fig/05_artefatos.png", dpi=140); plt.show()
crono.marco("simulação de artefatos de RM")

### 4.3 · Montagem do conjunto de treino final

Real + sintético na mesma pasta, um único `data.yaml`. O `val` aponta para a partição
**interna** — é o que o Ultralytics vai usar para escolher o melhor checkpoint durante
o treino. As partições externa e OOD ficam intocadas até a seção 6, e é isso que
mantém a avaliação limpa.

In [ ]:
def montar_yaml(raiz, nome, treino_rel, val_rel, nomes):
    p = os.path.join(raiz, f"data_{nome}.yaml")
    bloco = "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(nomes))
    open(p, "w").write(f"path: {os.path.abspath(raiz)}\ntrain: {treino_rel}\n"
                       f"val: {val_rel}\n\n{bloco}")
    return p

FINAL_IMG = f"{BASE}/images/treino_final"
FINAL_LAB = f"{BASE}/labels/treino_final"
for d in (FINAL_IMG, FINAL_LAB):
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
for orig, dest, ext in ((f"{BASE}/images/treino", FINAL_IMG, "*"),
                        (f"{BASE}/labels/treino", FINAL_LAB, "*.txt")):
    for p in glob.glob(f"{orig}/{ext}"): shutil.copy(p, dest)
if cfg.usar_sinteticas:
    for p in glob.glob(f"{SINT_IMG}/*"):     shutil.copy(p, FINAL_IMG)
    for p in glob.glob(f"{SINT_LAB}/*.txt"): shutil.copy(p, FINAL_LAB)

YAML_TREINO = montar_yaml(BASE, "treino_final", "images/treino_final",
                          "images/interno", ["negative", "positive"])
print(f"Treino final: {len(os.listdir(FINAL_IMG))} imagens "
      f"({len(particoes['treino'])} reais + {len(sinteticas)} sintéticas)")
print(open(YAML_TREINO).read())
crono.marco("montagem do conjunto de treino")

---
## 5 · Treino sob orçamento fechado

Aqui está a decisão de engenharia mais importante do notebook.

O Colab sorteia o hardware. A mesma célula pode receber uma T4, uma L4 ou uma CPU, e a
mesma configuração de `epochs` produz tempos de execução que variam por um fator de
cinco. Um notebook que promete "30 minutos" com `epochs=50` está, na prática,
prometendo nada.

A solução é inverter a variável de controle. O Ultralytics aceita o argumento
`time=<horas>`, documentado assim: *"tempo máximo de treinamento em horas. Se definido,
sobrescreve o argumento `epochs`, permitindo que o treinamento pare automaticamente
após a duração especificada."* O treinador mede a duração real da primeira época,
recalcula quantas cabem no tempo, e encerra devolvendo o melhor checkpoint.

**Consequência:** o número de épocas passa a ser *saída* do experimento, não entrada.
Em GPU rápida o modelo treina mais; em GPU lenta, menos. O relógio, que é o que o
usuário sente, fica fixo.

*Verificado experimentalmente:* passando `epochs=2, time=0.05`, o treinador reprogramou
a execução para 55 épocas — ou seja, `time` realmente sobrescreve `epochs`.

**As outras escolhas, e o porquê de cada uma:**

| argumento | valor | motivo |
|---|---|---|
| `cache="ram"` | — | são ~1000 imagens pequenas; o disco do Colab é o gargalo, não a memória |
| `cos_lr=True` | — | decaimento suave; com número de épocas indeterminado, evita cair num degrau ruim |
| `close_mosaic=5` | — | desliga o mosaico no fim para o modelo ver imagens inteiras antes de parar |
| `deterministic=False` | — | reprodutibilidade total custaria velocidade que o orçamento não tem — **assunção declarada** |
| `patience=100` | — | o corte é o tempo, não a estagnação |

In [ ]:
def treinar(cfg, yaml_treino, projeto, nome_run, cronometro=None):
    """
    O argumento `time` do Ultralytics é o que garante o orçamento: ele
    sobrescreve `epochs` e encerra o treino ao atingir o tempo declarado,
    devolvendo o melhor checkpoint até ali. Sem ele, o tempo total do
    notebook dependeria do hardware sorteado pelo Colab.
    """
    from ultralytics import YOLO
    modelo = YOLO(cfg.modelo)
    modelo.train(
        data=yaml_treino, epochs=cfg.epocas_teto, time=cfg.orcamento_treino_min / 60,
        imgsz=cfg.imgsz, batch=cfg.lote, device=cfg.device, seed=cfg.sementes,
        deterministic=False, cache="ram", workers=2, patience=100,
        cos_lr=True, close_mosaic=5, plots=True, val=True,
        project=projeto, name=nome_run, exist_ok=True, verbose=True,
    )
    pasta = os.path.join(projeto, nome_run)
    if cronometro:
        cronometro.marco("treino YOLO26")
    return YOLO(os.path.join(pasta, "weights", "best.pt")), pasta

modelo, PASTA_RUN = treinar(cfg, YAML_TREINO, os.path.join(RAIZ, "runs"),
                            f"yolo26_{MODO}", cronometro=crono)
print("\nMelhor checkpoint:", os.path.join(PASTA_RUN, "weights", "best.pt"))

### 5.1 · O que aconteceu durante o treino

Três painéis, três perguntas. **Esquerda:** as perdas de treino caem? **Centro:** as
perdas de validação acompanham, ou descolam (sobreajuste)? **Direita:** as métricas
subiram e estabilizaram, ou ainda estavam subindo quando o relógio zerou?

Se a curva da direita ainda está claramente em ascensão no fim, a leitura correta não
é "o modelo é ruim" — é "**o orçamento foi o gargalo**". Vale registrar isso na
conclusão em vez de esconder.

In [ ]:
csv_res = os.path.join(PASTA_RUN, "results.csv")
if os.path.exists(csv_res):
    df_treino = pd.read_csv(csv_res); df_treino.columns = [c.strip() for c in df_treino.columns]
    print(f"Épocas efetivamente concluídas: {int(df_treino['epoch'].max())}")
    display(df_treino.tail(3))
    fig_curvas_treino(csv_res, "fig/06_curvas_treino.png"); plt.show()
crono.marco("figuras de treino")

---
## 6 · Diagnóstico do modelo

Reportar um número de mAP e encerrar é o padrão dos trabalhos de aula. Mas mAP é uma
média de médias: ele diz *quanto*, nunca *onde* nem *por quê*. Esta seção troca um
número por seis perguntas.

| # | pergunta | instrumento |
|---|---|---|
| 6.1 | Quão bem, no total? | mAP@50 e mAP@50-95 |
| 6.2 | Com qual limiar devo operar? | varredura de τ e curva PR |
| 6.3 | Posso acreditar na confiança que ele reporta? | ECE e diagrama de confiabilidade |
| 6.4 | Que **tipo** de erro ele comete? | taxonomia estilo TIDE + matriz de confusão |
| 6.5 | Ele sobrevive a outro aparelho? | partição externa |
| 6.6 | Ele inventa achado onde não há nada? | robustez + controle OOD |

### 6.1 · Coleta de predições e recálculo das métricas

O Ultralytics já reporta mAP no `val()`. Ainda assim recalculamos tudo a partir das
predições cruas. Não é desconfiança: é que **as seis perguntas acima precisam dos
mesmos dados brutos**, e reaproveitar uma única coleta é o que mantém a seção dentro
do orçamento — além de tornar visível a matemática que a biblioteca esconde.

Um detalhe do YOLO26: como a cabeça é *end-to-end*, duplicatas deveriam ser raras.
"Deveriam" não é garantia, então há um passo explícito de **consolidação** por IoU —
que também torna o laudo determinístico.

In [ ]:
import os, glob, math, json
import numpy as np

# --------------------------------------------------------------- IoU
def iou_matriz(a, b):
    """a:(N,4) b:(M,4) em xyxy -> (N,M)."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    a = np.asarray(a, np.float32); b = np.asarray(b, np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2]-a[:, 0]) * (a[:, 3]-a[:, 1])
    ab = (b[:, 2]-b[:, 0]) * (b[:, 3]-b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

# --------------------------------------------- consolidação de detecções
def consolidar(caixas, confs, classes, iou_thr=0.55, agnostico=True):
    """
    Supressão gulosa de detecções redundantes.
    A cabeça do YOLO26 é end-to-end (sem NMS); ainda assim, modelos pouco
    treinados podem emitir duplicatas. Esta etapa deixa o laudo determinístico.
    """
    if len(caixas) == 0:
        return np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)
    caixas = np.asarray(caixas, np.float32); confs = np.asarray(confs, np.float32)
    classes = np.asarray(classes, int)
    ordem = np.argsort(-confs)
    mantidas = []
    for i in ordem:
        ok = True
        for j in mantidas:
            if not agnostico and classes[i] != classes[j]:
                continue
            if iou_matriz(caixas[i:i+1], caixas[j:j+1])[0, 0] > iou_thr:
                ok = False; break
        if ok:
            mantidas.append(i)
    m = np.array(mantidas, int)
    return caixas[m], confs[m], classes[m]

**Casamento guloso.** Ordenamos as predições por confiança decrescente; cada uma
disputa a caixa verdadeira de maior IoU ainda livre. Passou de `iou_tp=0.5` com a
classe certa, é acerto. Não passou, o *motivo* é registrado — e é o motivo, não a
contagem, que ensina alguma coisa. A taxonomia segue o TIDE (Bolya et al., 2020).

In [ ]:
# ------------------------------------------------------------ casamento
CATEGORIAS_ERRO = ["classificacao", "localizacao", "cls+loc", "duplicata", "fundo"]

def casar_imagem(gt_cx, gt_cls, pr_cx, pr_cf, pr_cls, iou_tp=0.5, iou_loc=0.1):
    """
    Casamento guloso por confiança decrescente.
    Devolve, por predição, o rótulo TP/erro e, por GT, se foi encontrado.
    Taxonomia inspirada no TIDE (Bolya et al., 2020).
    """
    ordem = np.argsort(-np.asarray(pr_cf)) if len(pr_cf) else np.array([], int)
    usados = set(); resultado = []
    M = iou_matriz(pr_cx, gt_cx) if len(pr_cx) and len(gt_cx) else np.zeros((len(pr_cx), len(gt_cx)))
    for i in ordem:
        linha = M[i] if M.size else np.zeros(0)
        if linha.size == 0:
            resultado.append((i, "FP", "fundo", -1, 0.0)); continue
        j = int(np.argmax(linha)); v = float(linha[j])
        mesma = (gt_cls[j] == pr_cls[i])
        if v >= iou_tp and mesma and j not in usados:
            usados.add(j); resultado.append((i, "TP", None, j, v))
        elif v >= iou_tp and mesma and j in usados:
            resultado.append((i, "FP", "duplicata", j, v))
        elif v >= iou_tp and not mesma:
            resultado.append((i, "FP", "classificacao", j, v))
        elif iou_loc <= v < iou_tp and mesma:
            resultado.append((i, "FP", "localizacao", j, v))
        elif iou_loc <= v < iou_tp and not mesma:
            resultado.append((i, "FP", "cls+loc", j, v))
        else:
            resultado.append((i, "FP", "fundo", j, v))
    faltantes = [j for j in range(len(gt_cx)) if j not in usados]
    return resultado, faltantes

**AP por interpolação em todos os pontos.** A precisão é tornada monótona
não-crescente e integramos a curva:

$$p_{\text{interp}}(r) = \max_{\tilde r \ge r} p(\tilde r), \qquad
  \mathrm{AP} = \sum_i (r_{i+1}-r_i)\, p_{\text{interp}}(r_{i+1})$$

O **mAP@50-95** repete isso para dez limiares de IoU (0,50 até 0,95, passo 0,05) e
tira a média. É por isso que ele sempre parece "pior": ele exige caixas cada vez mais
justas, não só objetos certos.

In [ ]:
# ---------------------------------------------------- AP / curvas PR
def _ap_todos_pontos(rec, prec):
    """AP por interpolação em todos os pontos (convenção COCO/VOC2010+)."""
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([1.0], prec, [0.0]))
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])), mrec, mpre

def curvas_por_classe(registros, n_classes, iou_tp=0.5):
    """
    registros: lista por imagem de dict(gt_cx, gt_cls, pr_cx, pr_cf, pr_cls)
    Devolve, por classe: recall, precision, AP, e vetores conf/TP para calibração.
    """
    saida = {}
    for c in range(n_classes):
        confs, tps, n_gt = [], [], 0
        for r in registros:
            gt_m = [k for k, v in enumerate(r["gt_cls"]) if v == c]
            n_gt += len(gt_m)
            gt_cx = [r["gt_cx"][k] for k in gt_m]
            pr_m = [k for k, v in enumerate(r["pr_cls"]) if v == c]
            if not pr_m:
                continue
            pr_cx = [r["pr_cx"][k] for k in pr_m]
            pr_cf = [r["pr_cf"][k] for k in pr_m]
            M = iou_matriz(pr_cx, gt_cx)
            usados = set()
            for i in np.argsort(-np.asarray(pr_cf)):
                confs.append(float(pr_cf[i]))
                if M.size == 0:
                    tps.append(0); continue
                j = int(np.argmax(M[i])); v = float(M[i, j])
                if v >= iou_tp and j not in usados:
                    usados.add(j); tps.append(1)
                else:
                    tps.append(0)
        confs = np.asarray(confs); tps = np.asarray(tps)
        if len(confs) == 0 or n_gt == 0:
            saida[c] = dict(ap=0.0, recall=np.zeros(1), precision=np.zeros(1),
                            confs=confs, tps=tps, n_gt=n_gt)
            continue
        o = np.argsort(-confs); confs, tps = confs[o], tps[o]
        tp_ac = np.cumsum(tps); fp_ac = np.cumsum(1 - tps)
        rec = tp_ac / n_gt
        prec = tp_ac / np.maximum(tp_ac + fp_ac, 1e-9)
        ap, _, _ = _ap_todos_pontos(rec, prec)
        saida[c] = dict(ap=ap, recall=rec, precision=prec, confs=confs,
                        tps=tps, n_gt=n_gt)
    return saida

def mapa(registros, n_classes, ious=None):
    """mAP@50 e mAP@50:95 calculados a partir dos mesmos registros."""
    ious = ious if ious is not None else np.arange(0.5, 1.0, 0.05)
    por_iou = []
    for t in ious:
        c = curvas_por_classe(registros, n_classes, iou_tp=float(t))
        por_iou.append(np.mean([c[k]["ap"] for k in c]))
    return dict(map50=float(por_iou[0]), map50_95=float(np.mean(por_iou)),
                por_iou=dict(zip([round(float(t), 2) for t in ious], por_iou)))

In [ ]:
import os, glob, json, math, datetime
import numpy as np
import cv2
import sys

VERSAO_LAUDO = "1.0"

# ------------------------------------------------------ coleta em lote
def coletar(modelo, caminhos_img, raiz_labels=None, conf_min=0.05, imgsz=640,
            iou_consolidacao=0.55, lote=16, device=None, augment=False):
    """
    Roda o modelo sobre uma lista de imagens e devolve `registros`
    no formato consumido por c_avaliacao (GT em pixels, predição em pixels).
    conf_min baixo de propósito: as curvas PR precisam da cauda de baixa confiança.
    """
    registros = []
    for i in range(0, len(caminhos_img), lote):
        bloco = caminhos_img[i:i + lote]
        preds = modelo.predict(bloco, imgsz=imgsz, conf=conf_min, verbose=False,
                               device=device, augment=augment)
        for cam, p in zip(bloco, preds):
            H, W = p.orig_shape
            b = p.boxes
            cx = b.xyxy.cpu().numpy() if len(b) else np.zeros((0, 4))
            cf = b.conf.cpu().numpy() if len(b) else np.zeros(0)
            cl = b.cls.cpu().numpy().astype(int) if len(b) else np.zeros(0, int)
            cx, cf, cl = consolidar(cx, cf, cl, iou_thr=iou_consolidacao)
            gt_cx, gt_cls = [], []
            if raiz_labels:
                lp = os.path.join(raiz_labels,
                                  os.path.splitext(os.path.basename(cam))[0] + ".txt")
                if os.path.exists(lp):
                    for l in open(lp).read().strip().split("\n"):
                        if not l.strip():
                            continue
                        q = l.split()
                        gt_cls.append(int(q[0]))
                        gt_cx.append(yolo_para_pixel(*[float(v) for v in q[1:5]], W, H))
            registros.append(dict(caminho=cam, largura=W, altura=H,
                                  gt_cx=gt_cx, gt_cls=gt_cls,
                                  pr_cx=cx.tolist(), pr_cf=cf.tolist(),
                                  pr_cls=cl.tolist()))
    return registros

In [ ]:
IMGS_INT = sorted(glob.glob(f"{BASE}/images/interno/*"))
IMGS_EXT = sorted(glob.glob(f"{BASE}/images/externo/*"))

regs_int = coletar(modelo, IMGS_INT, raiz_labels=f"{BASE}/labels/interno",
                   conf_min=cfg.conf_coleta, imgsz=cfg.imgsz,
                   iou_consolidacao=cfg.iou_consolidacao, device=cfg.device)
regs_ext = coletar(modelo, IMGS_EXT, raiz_labels=f"{BASE}/labels/externo",
                   conf_min=cfg.conf_coleta, imgsz=cfg.imgsz,
                   iou_consolidacao=cfg.iou_consolidacao, device=cfg.device)

NOMES = ["negative", "positive"]
m_int = mapa(regs_int, 2); m_ext = mapa(regs_ext, 2)
print(f"INTERNO  mAP@50 = {m_int['map50']:.4f}   mAP@50-95 = {m_int['map50_95']:.4f}")
print(f"EXTERNO  mAP@50 = {m_ext['map50']:.4f}   mAP@50-95 = {m_ext['map50_95']:.4f}")
crono.marco("coleta de predições (interno + externo)")

### 6.2 · Escolher o limiar: não existe τ universal

Um detector não devolve decisões, devolve **pontuações**. A decisão nasce quando
alguém escolhe um corte τ — e essa escolha é um julgamento de valor, não um detalhe
técnico.

Duas políticas, dois τ diferentes:

- **τ\* de F1** — maximiza a média harmônica entre precisão e recall. É o τ de quem
  quer o melhor equilíbrio agregado.
- **τ de triagem** — o **maior** τ que ainda garante recall ≥ 0,85. É o τ de quem
  prefere revisar falsos positivos a perder uma lesão.

No contexto simulado aqui, a segunda política é a defensável: o custo de um falso
negativo não é comparável ao de um falso positivo. O laudo da seção 7 usa esse τ.

In [ ]:
# ------------------------------------------------- ponto de operação
def varredura_limiar(registros, n_classes, iou_tp=0.5, limiares=None):
    """P, R e F1 globais (micro) em função do limiar de confiança."""
    limiares = limiares if limiares is not None else np.round(np.arange(0.05, 0.96, 0.01), 2)
    linhas = []
    for t in limiares:
        TP = FP = FN = 0
        for r in registros:
            keep = [i for i, c in enumerate(r["pr_cf"]) if c >= t]
            pr_cx = [r["pr_cx"][i] for i in keep]
            pr_cf = [r["pr_cf"][i] for i in keep]
            pr_cls = [r["pr_cls"][i] for i in keep]
            res, falt = casar_imagem(r["gt_cx"], r["gt_cls"], pr_cx, pr_cf, pr_cls, iou_tp)
            TP += sum(1 for x in res if x[1] == "TP")
            FP += sum(1 for x in res if x[1] == "FP")
            FN += len(falt)
        P = TP / max(TP + FP, 1e-9); R = TP / max(TP + FN, 1e-9)
        F1 = 2 * P * R / max(P + R, 1e-9)
        linhas.append(dict(limiar=float(t), TP=TP, FP=FP, FN=FN,
                           precisao=P, recall=R, f1=F1))
    return linhas

def escolher_ponto_operacional(varredura, recall_minimo=0.85):
    """
    Duas políticas:
      - 'f1'      : maximiza F1 (equilíbrio)
      - 'triagem' : maior limiar que ainda garante recall >= recall_minimo
                    (rastreio clínico prioriza não perder lesão)
    """
    melhor_f1 = max(varredura, key=lambda l: l["f1"])
    viaveis = [l for l in varredura if l["recall"] >= recall_minimo]
    triagem = max(viaveis, key=lambda l: l["limiar"]) if viaveis else None
    return dict(f1=melhor_f1, triagem=triagem, recall_minimo=recall_minimo)

In [ ]:
curvas_int = curvas_por_classe(regs_int, 2)
varredura  = varredura_limiar(regs_int, 2)
ponto      = escolher_ponto_operacional(varredura, cfg.recall_minimo_triagem)

for c, d in curvas_int.items():
    print(f"  {NOMES[c]:<9} AP@50 = {d['ap']:.4f}   ({d['n_gt']} caixas verdadeiras)")
print(f"\nτ* (máximo F1) ....: {ponto['f1']['limiar']:.2f}  "
      f"P={ponto['f1']['precisao']:.3f} R={ponto['f1']['recall']:.3f} F1={ponto['f1']['f1']:.3f}")
if ponto["triagem"]:
    TAU = ponto["triagem"]["limiar"]
    print(f"τ  (triagem, R≥{cfg.recall_minimo_triagem:.2f}): {TAU:.2f}  "
          f"P={ponto['triagem']['precisao']:.3f} R={ponto['triagem']['recall']:.3f}")
else:
    TAU = ponto["f1"]["limiar"]
    print(f"τ  (triagem) ......: inatingível — o modelo nunca alcança "
          f"recall {cfg.recall_minimo_triagem:.2f}. Usando τ* de F1 = {TAU:.2f}.")
print(f"\n➜ Limiar operacional adotado: τ = {TAU:.2f}")

fig_pr_e_limiar(curvas_int, varredura, ponto, NOMES, "fig/07_pr_limiar.png"); plt.show()
crono.marco("curvas PR e ponto de operação")

### 6.3 · A confiança reportada significa alguma coisa?

Quando o modelo diz `0,80`, ele acerta 80% das vezes? Um modelo **calibrado** sim. O
**Expected Calibration Error** mede o desvio, agrupando as detecções em $M$ faixas de
confiança:

$$\mathrm{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{n}\,\bigl|\,\mathrm{acc}(B_m) - \mathrm{conf}(B_m)\,\bigr|$$

No diagrama de confiabilidade, barras **abaixo** da diagonal indicam superconfiança
(o caso comum em redes profundas modernas); **acima**, subconfiança. Importa porque o
laudo da seção 7 imprime esse número para uma pessoa ler — e um número superconfiante
impresso num laudo é pior do que nenhum número.

In [ ]:
# --------------------------------------------------------- calibração
def ece(confs, acertos, n_bins=10):
    """Expected Calibration Error + dados do diagrama de confiabilidade."""
    confs = np.asarray(confs, float); acertos = np.asarray(acertos, float)
    if len(confs) == 0:
        return 0.0, []
    bordas = np.linspace(0, 1, n_bins + 1)
    erro, linhas = 0.0, []
    for i in range(n_bins):
        m = (confs > bordas[i]) & (confs <= bordas[i + 1])
        if m.sum() == 0:
            linhas.append(dict(centro=(bordas[i]+bordas[i+1])/2, n=0,
                               conf_media=np.nan, acuracia=np.nan)); continue
        cm, ac = confs[m].mean(), acertos[m].mean()
        erro += m.sum() / len(confs) * abs(ac - cm)
        linhas.append(dict(centro=(bordas[i]+bordas[i+1])/2, n=int(m.sum()),
                           conf_media=float(cm), acuracia=float(ac)))
    return float(erro), linhas

In [ ]:
confs_all = np.concatenate([curvas_int[c]["confs"] for c in curvas_int])
tps_all   = np.concatenate([curvas_int[c]["tps"]   for c in curvas_int])
valor_ece, faixas = ece(confs_all, tps_all, n_bins=10)
print(f"ECE = {valor_ece:.4f}  (0 = perfeitamente calibrado)")
display(pd.DataFrame(faixas).round(3))
fig_calibracao(faixas, valor_ece, "fig/08_calibracao.png"); plt.show()
crono.marco("análise de calibração")

### 6.4 · A anatomia do erro

Dois modelos com o mesmo mAP podem estar errando de formas completamente diferentes —
e as correções são opostas. Categorias:

| categoria | o que aconteceu | o que fazer |
|---|---|---|
| **classe errada (IoU ok)** | achou a lesão, errou `negative`/`positive` | mais dado rotulado, revisar ambiguidade do rótulo |
| **caixa mal posicionada** | 0,1 ≤ IoU < 0,5 | regressão fraca: mais resolução, mais épocas |
| **invenção sobre o fundo** | IoU < 0,1 com qualquer verdade | mais imagens negativas, τ mais alto |
| **duplicata** | segunda caixa no mesmo alvo | consolidação / cabeça one-to-one |
| **lesão não detectada** | nenhuma predição cobriu a verdade | o erro caro em triagem |

In [ ]:
# ------------------------------------------------------ matriz de confusão
def matriz_confusao(registros, n_classes, nomes, limiar=0.25, iou_tp=0.5):
    """(n+1)x(n+1): última linha/coluna = fundo (FP de fundo / GT não detectado)."""
    n = n_classes
    M = np.zeros((n + 1, n + 1), int)   # [predito, verdadeiro]
    for r in registros:
        keep = [i for i, c in enumerate(r["pr_cf"]) if c >= limiar]
        pr_cx = [r["pr_cx"][i] for i in keep]
        pr_cf = [r["pr_cf"][i] for i in keep]
        pr_cls = [r["pr_cls"][i] for i in keep]
        Mi = iou_matriz(pr_cx, r["gt_cx"]) if pr_cx and len(r["gt_cx"]) else np.zeros((len(pr_cx), len(r["gt_cx"])))
        usados = set()
        for i in np.argsort(-np.asarray(pr_cf)) if pr_cf else []:
            if Mi.size == 0:
                M[pr_cls[i], n] += 1; continue
            j = int(np.argmax(Mi[i])); v = float(Mi[i, j])
            if v >= iou_tp and j not in usados:
                usados.add(j); M[pr_cls[i], r["gt_cls"][j]] += 1
            else:
                M[pr_cls[i], n] += 1
        for j in range(len(r["gt_cx"])):
            if j not in usados:
                M[n, r["gt_cls"][j]] += 1
    rotulos = list(nomes) + ["fundo"]
    return M, rotulos

def taxonomia_erros(registros, limiar=0.25, iou_tp=0.5):
    cont = {k: 0 for k in CATEGORIAS_ERRO}
    tp = faltas = 0
    for r in registros:
        keep = [i for i, c in enumerate(r["pr_cf"]) if c >= limiar]
        res, falt = casar_imagem(r["gt_cx"], r["gt_cls"],
                                 [r["pr_cx"][i] for i in keep],
                                 [r["pr_cf"][i] for i in keep],
                                 [r["pr_cls"][i] for i in keep], iou_tp)
        tp += sum(1 for x in res if x[1] == "TP")
        for x in res:
            if x[1] == "FP":
                cont[x[2]] += 1
        faltas += len(falt)
    cont_total = dict(cont); cont_total["nao_detectado"] = faltas
    return dict(tp=tp, erros=cont_total, total_fp=sum(cont.values()))

In [ ]:
M_conf, rot_conf = matriz_confusao(regs_int, 2, NOMES, limiar=TAU)
tax = taxonomia_erros(regs_int, limiar=TAU)
print("Acertos (TP):", tax["tp"], " | Falsos positivos:", tax["total_fp"])
display(pd.DataFrame([tax["erros"]]).T.rename(columns={0: "ocorrências"}))
fig_matriz_confusao(M_conf, rot_conf, "fig/09_matriz_confusao.png",
                    f"Matriz de confusão · partição interna · τ={TAU:.2f}"); plt.show()
fig_taxonomia(tax, "fig/10_taxonomia.png"); plt.show()
crono.marco("matriz de confusão e taxonomia de erros")

### 6.5 · Generalização: o preço do deslocamento de domínio

Agora a partição **externa** — 175 imagens de um protocolo que o modelo **nunca viu**,
com uma prevalência de classe completamente diferente (84% `positive` contra 46% no
treino).

A queda de mAP entre interno e externo é a estimativa mais honesta que este notebook
consegue produzir do que aconteceria se o sistema recebesse imagens de outro aparelho.
Se a queda for grande, a conclusão não é "o modelo é ruim": é que **o número reportado
no split interno estava superestimando a realidade** — que é exatamente o que a maioria
dos trabalhos publica sem perceber.

In [ ]:
var_ext   = varredura_limiar(regs_ext, 2)
por_lim   = {l["limiar"]: l for l in var_ext}
ponto_ext = por_lim.get(round(TAU, 2), min(var_ext, key=lambda l: abs(l["limiar"] - TAU)))
ponto_int = min(varredura, key=lambda l: abs(l["limiar"] - TAU))

comparativo = [
    dict(nome="interno (in-dist.)", map50=m_int["map50"], map50_95=m_int["map50_95"],
         precisao=ponto_int["precisao"], recall=ponto_int["recall"]),
    dict(nome="externo (domain shift)", map50=m_ext["map50"], map50_95=m_ext["map50_95"],
         precisao=ponto_ext["precisao"], recall=ponto_ext["recall"]),
]
display(pd.DataFrame(comparativo).round(4))
queda = 100 * (1 - m_ext["map50"] / max(m_int["map50"], 1e-9))
print(f"\n➜ Queda de mAP@50 sob deslocamento de protocolo: {queda:.1f}%")
fig_dominios(comparativo, "fig/11_dominios.png"); plt.show()
crono.marco("validação externa (domain shift)")

### 6.6 · Robustez a artefatos e alarme falso fora do domínio

Duas perguntas de segurança que raramente entram num relatório de aula.

**(a) Robustez.** As imagens da partição interna passam pelos quatro artefatos da seção
4.2, em três severidades. A curva de mAP em função da severidade mostra a qual
degradação o modelo é mais frágil — e ruído Riciano forte tende a ser o pior caso,
porque destrói exatamente a textura que distingue a lesão do parênquima.

**(b) Alarme falso fora do domínio.** As 115 imagens de `medical-pills`. Nenhuma contém
encéfalo. Toda caixa emitida ali é invenção pura, e a **taxa de imagens com alarme** é
uma métrica de segurança direta: é a frequência com que o sistema entregaria um laudo
positivo para uma entrada que jamais deveria ter sido processada.

In [ ]:
import os, glob, shutil, sys
import cv2, numpy as np

def preparar_corrompido(imagens, raiz_labels, destino, artefato, severidade):
    di = os.path.join(destino, "images"); dl = os.path.join(destino, "labels")
    os.makedirs(di, exist_ok=True); os.makedirs(dl, exist_ok=True)
    for ip in imagens:
        nome = os.path.basename(ip); stem = os.path.splitext(nome)[0]
        img = carregar_cinza(ip)
        cv2.imwrite(os.path.join(di, nome), aplicar_artefato(img, artefato, severidade))
        lp = os.path.join(raiz_labels, stem + ".txt")
        shutil.copy(lp, os.path.join(dl, stem + ".txt")) if os.path.exists(lp) \
            else open(os.path.join(dl, stem + ".txt"), "w").close()
    return sorted(glob.glob(di + "/*")), dl

def avaliar_robustez(modelo, imagens, raiz_labels, cfg, n_classes=2,
                     artefatos=None, severidades=(1, 2, 3), tmp="/tmp/robustez"):
    """
    Devolve {artefato: {0: mAP50_original, 1: ..., 2: ..., 3: ...}}.
    A severidade 0 é sempre a imagem intacta (mesma amostra, base comparável).
    """
    artefatos = artefatos or list(ARTEFATOS)
    base = coletar(modelo, imagens, raiz_labels=raiz_labels,
                   conf_min=cfg.conf_coleta, imgsz=cfg.imgsz,
                   iou_consolidacao=cfg.iou_consolidacao, device=cfg.device)
    m0 = mapa(base, n_classes, ious=[0.5])["map50"]
    tabela = {}
    for a in artefatos:
        tabela[a] = {0: m0}
        for s in severidades:
            d = os.path.join(tmp, f"{a}_{s}")
            shutil.rmtree(d, ignore_errors=True)
            imgs_c, lab_c = preparar_corrompido(imagens, raiz_labels, d, a, s)
            regs = coletar(modelo, imgs_c, raiz_labels=lab_c,
                           conf_min=cfg.conf_coleta, imgsz=cfg.imgsz,
                           iou_consolidacao=cfg.iou_consolidacao, device=cfg.device)
            tabela[a][s] = mapa(regs, n_classes, ious=[0.5])["map50"]
    return tabela, m0

def taxa_alarme_falso_ood(modelo, imagens_ood, limiar, cfg):
    """
    Controle negativo: imagens médicas de OUTRO domínio (medical-pills), onde
    nenhum achado é possível. Mede com que frequência o pipeline inventa lesão.
    """
    regs = coletar(modelo, imagens_ood, raiz_labels=None, conf_min=limiar,
                   imgsz=cfg.imgsz, iou_consolidacao=cfg.iou_consolidacao,
                   device=cfg.device)
    com_alarme = sum(1 for r in regs if len(r["pr_cf"]) > 0)
    total_caixas = sum(len(r["pr_cf"]) for r in regs)
    confs = [c for r in regs for c in r["pr_cf"]]
    return dict(imagens=len(regs), imagens_com_alarme=com_alarme,
                taxa_alarme_falso=com_alarme / max(len(regs), 1),
                caixas_espurias=total_caixas,
                caixas_por_imagem=total_caixas / max(len(regs), 1),
                confianca_media=float(np.mean(confs)) if confs else 0.0,
                confianca_maxima=float(np.max(confs)) if confs else 0.0)

In [ ]:
amostra_rob = IMGS_INT[:cfg.artefatos_amostra]
tabela_rob, map_base = avaliar_robustez(modelo, amostra_rob, f"{BASE}/labels/interno",
                                        cfg, n_classes=2, severidades=cfg.severidades)
display(pd.DataFrame(tabela_rob).T.round(4).rename_axis("artefato")
        .rename(columns={0: "original", 1: "sev 1", 2: "sev 2", 3: "sev 3"}))
fig_robustez(tabela_rob, "fig/12_robustez.png"); plt.show()

ood = taxa_alarme_falso_ood(modelo, OOD, TAU, cfg)
print("\nControle negativo (medical-pills, fora do domínio):")
for k, v in ood.items():
    print(f"  {k:.<30} {v:.4f}" if isinstance(v, float) else f"  {k:.<30} {v}")
crono.marco("robustez + controle negativo OOD")

### 6.7 · Ver os erros com os próprios olhos

Nenhuma métrica substitui olhar. Verde é acerto, magenta é falso positivo, laranja
tracejado é lesão que o modelo perdeu — as cores de alerta do próprio template. Um erro que você consegue explicar olhando é
um erro que você sabe como corrigir.

In [ ]:
fig_qualitativa(regs_int, NOMES, limiar=TAU, n=6,
                caminho="fig/13_qualitativa_interno.png", semente=cfg.sementes); plt.show()
fig_qualitativa(regs_ext, NOMES, limiar=TAU, n=6,
                caminho="fig/14_qualitativa_externo.png", semente=cfg.sementes); plt.show()
crono.marco("inspeção qualitativa")

---
## 7 · Do tensor ao laudo

Até aqui o modelo devolve `xyxy`, `conf`, `cls`. Ninguém lê isso. A última milha de
qualquer sistema de visão é transformar tensor em **informação que uma pessoa
interpreta** — e é aqui que a maioria dos projetos de aula para.

O motor de laudo calcula, para cada achado:

| campo | como é obtido | assunção declarada |
|---|---|---|
| área, Ø equivalente | $A = w\cdot h$, $D = 2\sqrt{A/\pi}$ | a caixa aproxima a lesão |
| área relativa | $A / A_{\text{intracraniana}}$ | máscara de Otsu, não segmentação clínica |
| lateralidade | posição em relação à linha média | **linha média = centróide horizontal**; é lateralidade *da imagem*, não anatômica (a convenção radiológica depende do DICOM) |
| contraste relativo | $\bar I_{\text{lesão}} / \bar I_{\text{encéfalo}}$ | intensidade em JPEG, não em unidades físicas |
| estabilidade | ver abaixo | proxy, **não** probabilidade |
| medidas em mm | opcional | **só** se o `PixelSpacing` do DICOM for informado |

**Sobre a estabilidade.** O YOLO26 não aceita `augment=True` — a cabeça end-to-end não
suporta test-time augmentation (verificado: o Ultralytics emite
`Model does not support 'augment=True'` e volta para escala única). Então a incerteza é
estimada de outro jeito: a inferência é repetida com a imagem **espelhada** (as caixas
voltam ao referencial original) e em **outra escala de entrada**, e mede-se o IoU médio
entre as detecções originais e as das duas vistas.

Um achado com estabilidade 0,95 aparece igual sob qualquer transformação. Um achado com
0,30 é uma detecção que depende de como você olhou — e um laudo honesto precisa dizer
isso. **Não é uma probabilidade calibrada**, é um indicador de concordância.

E uma medida que **não** está aqui de propósito: nada em milímetros, a menos que
alguém informe a escala. JPEG não carrega `PixelSpacing`. Inventar milímetros a partir
de pixels seria a mentira mais fácil e mais perigosa deste notebook.

In [ ]:
# --------------------------------------------------- métricas de imagem
def _linha_media(masc):
    """Estimativa da linha média sagital: centróide horizontal do encéfalo."""
    ys, xs = np.where(masc > 0)
    return float(xs.mean()) if len(xs) else masc.shape[1] / 2

def _descrever_lesao(img, masc, caixa, area_encefalo, x_media, idx, classe,
                     nome_classe, conf):
    x1, y1, x2, y2 = [float(v) for v in caixa]
    w, h = x2 - x1, y2 - y1
    area = w * h
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    recorte = img[int(max(0, y1)):int(y2), int(max(0, x1)):int(x2)]
    intensidade = float(recorte.mean()) if recorte.size else float("nan")
    dentro = img[masc > 0]
    ref = float(dentro.mean()) if dentro.size else float(img.mean())
    return {
        "id": f"L{idx+1:02d}",
        "achado": nome_classe,
        "classe_id": int(classe),
        "confianca": round(float(conf), 4),
        "caixa_px": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)],
        "centro_px": [round(cx, 1), round(cy, 1)],
        "largura_px": round(w, 1), "altura_px": round(h, 1),
        "area_px2": round(area, 1),
        "area_relativa_encefalo_pct": round(100 * area / area_encefalo, 3) if area_encefalo else None,
        "diametro_equivalente_px": round(2 * math.sqrt(area / math.pi), 1),
        "razao_aspecto": round(w / h, 3) if h else None,
        "lateralidade_imagem": "esquerda da imagem" if cx < x_media else "direita da imagem",
        "deslocamento_da_linha_media_px": round(cx - x_media, 1),
        "intensidade_media": round(intensidade, 1),
        "contraste_relativo": round(intensidade / ref, 3) if ref else None,
    }

def gerar_laudo(modelo, caminho_img, nomes_classes, limiar=0.25, imgsz=640,
                iou_consolidacao=0.55, device=None, consistencia=True,
                escala_mm_por_px=None):
    """
    Executa o pipeline completo numa única imagem e devolve um laudo estruturado.
    `escala_mm_por_px` é OPCIONAL: sem o cabeçalho DICOM não há como inferir a
    escala física, então as medidas saem em pixels por padrão.
    """
    img = carregar_cinza(caminho_img)
    H, W = img.shape
    masc = mascara_encefalo(img)
    area_encefalo = float((masc > 0).sum())
    x_media = _linha_media(masc)

    p = modelo.predict([caminho_img], imgsz=imgsz, conf=limiar, verbose=False,
                       device=device)[0]
    b = p.boxes
    cx = b.xyxy.cpu().numpy() if len(b) else np.zeros((0, 4))
    cf = b.conf.cpu().numpy() if len(b) else np.zeros(0)
    cl = b.cls.cpu().numpy().astype(int) if len(b) else np.zeros(0, int)
    cx, cf, cl = consolidar(cx, cf, cl, iou_thr=iou_consolidacao)

    # ---- consistência sob transformações (proxy honesto de incerteza)
    #      A cabeça end-to-end do YOLO26 não aceita `augment=True`; então a
    #      estabilidade é medida repetindo a inferência com (a) espelhamento
    #      horizontal e (b) outra escala de entrada, e casando as caixas por IoU.
    estabilidade = None
    if consistencia and len(cx):
        vistas = []
        # (a) espelhamento horizontal — também sonda viés esquerda/direita
        espelhada = cv2.flip(img, 1)
        tmp = os.path.join("/tmp", "_flip_" + os.path.basename(caminho_img))
        cv2.imwrite(tmp, espelhada)
        pf = modelo.predict([tmp], imgsz=imgsz, conf=limiar, verbose=False, device=device)[0]
        bf = pf.boxes
        if len(bf):
            cf_box = bf.xyxy.cpu().numpy().copy()
            cf_box[:, [0, 2]] = W - cf_box[:, [2, 0]]       # desfaz o espelhamento
            vistas.append(cf_box)
        else:
            vistas.append(np.zeros((0, 4)))
        try:
            os.remove(tmp)
        except OSError:
            pass
        # (b) outra escala de entrada
        alt = int(imgsz * 0.75) // 32 * 32
        pe = modelo.predict([caminho_img], imgsz=max(alt, 320), conf=limiar,
                            verbose=False, device=device)[0]
        vistas.append(pe.boxes.xyxy.cpu().numpy() if len(pe.boxes) else np.zeros((0, 4)))
        acumulado = []
        for v in vistas:
            acumulado.append(iou_matriz(cx, v).max(axis=1) if len(v) else np.zeros(len(cx)))
        estabilidade = [round(float(v), 3) for v in np.mean(acumulado, axis=0)]

    lesoes = []
    for i in range(len(cx)):
        d = _descrever_lesao(img, masc, cx[i], area_encefalo, x_media, i,
                             cl[i], nomes_classes[int(cl[i])], cf[i])
        if estabilidade is not None:
            d["estabilidade_iou"] = estabilidade[i]
        if escala_mm_por_px:
            d["diametro_equivalente_mm"] = round(
                d["diametro_equivalente_px"] * escala_mm_por_px, 2)
            d["area_mm2"] = round(d["area_px2"] * escala_mm_por_px ** 2, 1)
        lesoes.append(d)
    lesoes.sort(key=lambda d: -d["confianca"])
    for i, d in enumerate(lesoes):
        d["id"] = f"L{i+1:02d}"

    carga = sum(d["area_px2"] for d in lesoes)
    laudo = {
        "versao_laudo": VERSAO_LAUDO,
        "gerado_em": datetime.datetime.now().isoformat(timespec="seconds"),
        "arquivo": os.path.basename(caminho_img),
        "imagem": {"largura_px": W, "altura_px": H,
                   "area_intracraniana_px2": int(area_encefalo),
                   "linha_media_estimada_px": round(x_media, 1)},
        "parametros": {"limiar_confianca": limiar, "imgsz": imgsz,
                       "iou_consolidacao": iou_consolidacao, "teste_consistencia": bool(consistencia),
                       "escala_mm_por_px": escala_mm_por_px},
        "resumo": {
            "n_achados": len(lesoes),
            "veredito": "achado detectável" if lesoes else "sem achado acima do limiar",
            "confianca_maxima": round(float(max([d["confianca"] for d in lesoes])), 4) if lesoes else 0.0,
            "carga_lesional_px2": round(carga, 1),
            "carga_lesional_pct_encefalo": round(100 * carga / area_encefalo, 3) if area_encefalo else None,
            "classes_detectadas": sorted({d["achado"] for d in lesoes}),
        },
        "achados": lesoes,
        "aviso": ("Saída de um exercício acadêmico de visão computacional. "
                  "NÃO é dispositivo médico, NÃO tem validação clínica e "
                  "NÃO deve ser usada para decisão diagnóstica."),
    }
    return laudo, p

def laudo_em_texto(laudo):
    r = laudo["resumo"]; L = []
    L.append(f"LAUDO AUTOMÁTICO (v{laudo['versao_laudo']}) — {laudo['arquivo']}")
    L.append(f"Gerado em {laudo['gerado_em']}")
    L.append("-" * 64)
    L.append(f"Veredito.................: {r['veredito']}")
    L.append(f"Achados acima do limiar..: {r['n_achados']}  "
             f"(limiar τ = {laudo['parametros']['limiar_confianca']})")
    L.append(f"Confiança máxima.........: {r['confianca_maxima']:.3f}")
    L.append(f"Carga lesional...........: {r['carga_lesional_px2']:.0f} px² "
             f"({r['carga_lesional_pct_encefalo']}% da área intracraniana)")
    L.append(f"Classes..................: {', '.join(r['classes_detectadas']) or '—'}")
    L.append("-" * 64)
    for d in laudo["achados"]:
        L.append(f"[{d['id']}] {d['achado']}  conf={d['confianca']:.3f}"
                 + (f"  estabilidade={d['estabilidade_iou']:.2f}" if "estabilidade_iou" in d else ""))
        L.append(f"      caixa {d['caixa_px']}  centro {d['centro_px']}")
        L.append(f"      área {d['area_px2']:.0f} px² ({d['area_relativa_encefalo_pct']}% do encéfalo)"
                 f"  Ø equiv. {d['diametro_equivalente_px']:.0f} px")
        L.append(f"      {d['lateralidade_imagem']} (Δ linha média {d['deslocamento_da_linha_media_px']:+.0f} px)"
                 f"  contraste rel. {d['contraste_relativo']}")
    L.append("-" * 64)
    L.append(laudo["aviso"])
    return "\n".join(L)

In [ ]:
alvo = [r["caminho"] for r in regs_int if r["gt_cx"]][:1]
laudo, resultado = gerar_laudo(modelo, alvo[0], NOMES, limiar=TAU, imgsz=cfg.imgsz,
                               iou_consolidacao=cfg.iou_consolidacao,
                               device=cfg.device, consistencia=True)
print(laudo_em_texto(laudo))

fig, ax = plt.subplots(1, 2, figsize=(11.5, 5))
ax[0].imshow(cv2.imread(alvo[0], cv2.IMREAD_GRAYSCALE), cmap="gray")
ax[0].set_title("entrada", fontsize=10, color=PALETA["escuro"])
ax[1].imshow(cv2.cvtColor(resultado.plot(), cv2.COLOR_BGR2RGB))
ax[1].set_title(f"detecções · τ={TAU:.2f}", fontsize=10, color=PALETA["escuro"])
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig("fig/15_laudo_exemplo.png", dpi=150); plt.show()

### 7.1 · O laudo como estrutura de dados

O texto acima é apenas uma *renderização*. O que o pipeline produz de fato é um objeto
JSON versionado — que um sistema hospitalar consumiria, um banco de dados armazenaria e
uma auditoria compararia entre execuções. O campo `versao_laudo` existe justamente
para isso: quando o esquema mudar, laudos antigos continuam interpretáveis.

In [ ]:
print(json.dumps(laudo, ensure_ascii=False, indent=2)[:2200], "\n  ...")
with open("laudo_exemplo.json", "w", encoding="utf-8") as f:
    json.dump(laudo, f, ensure_ascii=False, indent=2)
print("\n➜ salvo em laudo_exemplo.json")

### 7.2 · Laudo em lote: o pipeline como produtor de tabela

Um laudo é útil para uma pessoa. Uma **tabela de laudos** é útil para o serviço inteiro:
dá para ordenar por carga lesional, filtrar por estabilidade baixa, priorizar fila de
revisão. É a diferença entre uma demonstração e um sistema.

In [ ]:
N_LOTE = 12
linhas = []
for cam in [r["caminho"] for r in regs_int][:N_LOTE]:
    L, _ = gerar_laudo(modelo, cam, NOMES, limiar=TAU, imgsz=cfg.imgsz,
                       iou_consolidacao=cfg.iou_consolidacao,
                       device=cfg.device, consistencia=False)
    linhas.append(dict(arquivo=L["arquivo"], veredito=L["resumo"]["veredito"],
                       n_achados=L["resumo"]["n_achados"],
                       conf_max=L["resumo"]["confianca_maxima"],
                       carga_pct=L["resumo"]["carga_lesional_pct_encefalo"],
                       classes=", ".join(L["resumo"]["classes_detectadas"]) or "—"))
df_laudos = pd.DataFrame(linhas).sort_values("carga_pct", ascending=False)
display(df_laudos)
df_laudos.to_csv("laudos_lote.csv", index=False)
crono.marco("laudos em lote")

---
## 8 · Interface web

O modelo é um artefato técnico. A interface é o que faz dele um sistema que alguém usa
sem abrir um notebook.

Esta seção não desenha uma interface do zero: ela monta a aplicação **sobre o template
Dtox** (Themefisher) — Bootstrap 4, tipografia Poppins, os gradientes
`#17FFD3 → #D3FC71` e `#17FFD3 → #23E3EE`, o azul `#008DEC` e a sombra
`0 35px 46px rgba(172,189,199,.28)`. As mesmas cores que você viu em todas as figuras
das seções anteriores: a identidade visual do projeto **é** a do template.

O servidor é **Flask**, e o back-end não reimplementa nada — ele chama exatamente as
mesmas funções `gerar_laudo`, `carregar_cinza` e `mascara_encefalo` que já rodaram na
seção 7. Uma interface que reimplementa a regra de negócio é uma segunda fonte de
verdade esperando para divergir da primeira.

**Três páginas:**

| rota | o que faz |
|---|---|
| `/` | apresenta o pipeline, as partições e a síntese, com as figuras desta execução |
| `/laudo` | envia uma imagem, ajusta τ e devolve o laudo completo + JSON para baixar |
| `/resultados` | lê o `resultados.json` desta execução e monta o painel de diagnóstico |
| `/metodo` | método, decisões de treino, eixos de avaliação e limitações |
| `/api/laudo` | o mesmo motor em JSON, para integrar com outro sistema |

E o aviso de que isto não é dispositivo médico fica fixo em todas as páginas, não
escondido em rodapé. Se o sistema é apresentado como se fosse clínico, a
responsabilidade muda de lugar.

### 8.1 · O pacote da interface

A célula abaixo escreve a aplicação em `neuro26/`: o `app.py` com as rotas, os cinco
templates Jinja e o CSS. O CSS é o do próprio Dtox — o `style.css` do template mais o
subconjunto do Bootstrap que acompanha o pacote —, recortado para as regras que estas
páginas usam e **não reescrito**.

Vai comprimido de propósito: são 200 KB de marcação e folha de estilo de terceiros, e
despejá-los em texto no meio de uma aula tornaria o notebook ilegível. Depois de rodar,
os arquivos ficam em `neuro26/` e podem ser abertos e editados normalmente.

> Se você tiver o pacote **completo** do Dtox (a pasta `webapp/` que acompanha esta
> entrega, com `static/dtox/` dentro), coloque-a ao lado do notebook: a aplicação detecta
> sozinha e passa a usar o template inteiro — imagens de fundo, formas decorativas,
> slick, AOS e venobox. Sem ela, roda em modo enxuto, com o mesmo layout.

In [ ]:
# O pacote da interface (app.py + templates + CSS do Dtox), comprimido.
import base64, gzip, io, tarfile, os, shutil

PACOTE_WEB = """...
IAMA"""

def escrever_interface(destino="neuro26"):
    """Descompacta a aplicação; devolve o caminho e o modo detectado."""
    shutil.rmtree(destino, ignore_errors=True)
    os.makedirs(destino, exist_ok=True)
    dados = gzip.decompress(base64.b64decode("".join(PACOTE_WEB.split())))
    with tarfile.open(fileobj=io.BytesIO(dados)) as tar:
        tar.extractall(destino)
    # se o pacote completo do Dtox estiver por perto, usa o template inteiro
    for origem in ("webapp/static/dtox", "../webapp/static/dtox", "dtox-1.0.0"):
        if os.path.isdir(origem):
            shutil.copytree(origem, os.path.join(destino, "static", "dtox"),
                            dirs_exist_ok=True)
            break
    completo = os.path.exists(os.path.join(destino, "static", "dtox", "css", "style.css"))
    return destino, completo

PASTA_WEB, MODO_COMPLETO = escrever_interface()
print("Aplicação escrita em", os.path.abspath(PASTA_WEB), "\n")
for a in ("app.py", "templates/base.html", "templates/index.html", "templates/laudo.html",
          "templates/resultados.html", "templates/metodo.html",
          "static/projeto/projeto.css", "static/projeto/projeto.js",
          "static/projeto/estilos.css"):
    cam = os.path.join(PASTA_WEB, a)
    print(f"  {a:<32} {os.path.getsize(cam)/1024:7.1f} KB")
if MODO_COMPLETO:
    n = sum(len(f) for _, _, f in os.walk(os.path.join(PASTA_WEB, "static", "dtox")))
    print(f"\nTemplate Dtox: pacote completo ({n} arquivos em static/dtox)")
else:
    print("\nTemplate Dtox: modo enxuto — subconjunto do style.css e do Bootstrap que"
          " acompanham o template")
crono.marco("escrita da interface")

### 8.2 · Como o back-end conversa com o pipeline

Vale ler o trecho abaixo antes de subir o servidor. A rota `/laudo` faz três coisas e
mais nada: decodifica a imagem, chama `gerar_laudo` — **a mesma função da seção 7** — e
desenha as caixas. Toda a matemática do laudo continua morando em um lugar só.

In [ ]:
# Mostra a rota principal, para leitura — o arquivo inteiro está em neuro26/app.py
fonte = open(f"{PASTA_WEB}/app.py", encoding="utf-8").read()
inicio = fonte.index('    @app.route("/laudo", methods=["GET", "POST"])')
fim = fonte.index('    @app.route("/api/laudo"')
print(fonte[inicio:fim])

### 8.3 · Subir o servidor

O Flask roda em uma thread separada para não travar o notebook. No Colab, a função
`google.colab.kernel.proxyPort` devolve um endereço navegável para a porta local —
é o caminho nativo, não exige token de serviço nenhum. Fora do Colab, o endereço é
o `localhost` de sempre.

In [ ]:
import sys, threading, time as _t
sys.path.insert(0, os.path.abspath(PASTA_WEB))
import importlib
if "app" in sys.modules:
    del sys.modules["app"]
import app as neuro26_web
importlib.reload(neuro26_web)

# As figuras desta execução entram na aplicação — o site mostra os SEUS resultados.
FIGURAS_SITE = {
    "composicao":  "fig/01_composicao.png",
    "geometria":   "fig/02_geometria.png",
    "calor":       "fig/03_mapa_calor.png",
    "sinteticas":  "fig/04_sinteticas.png",
    "artefatos":   "fig/05_artefatos.png",
    "treino":      "fig/06_curvas_treino.png",
    "pr_limiar":   "fig/07_pr_limiar.png",
    "calibracao":  "fig/08_calibracao.png",
    "matriz":      "fig/09_matriz_confusao.png",
    "taxonomia":   "fig/10_taxonomia.png",
    "dominios":    "fig/11_dominios.png",
    "robustez":    "fig/12_robustez.png",
    "qualitativa": "fig/13_qualitativa_interno.png",
    "orcamento":   "fig/16_orcamento.png",
}

aplicacao = neuro26_web.criar_app(
    pesos=os.path.join(PASTA_RUN, "weights", "best.pt"),
    dados=BASE, resultados=os.path.join(RAIZ, "resultados.json"),
    imgsz=cfg.imgsz, device=cfg.device,
    figuras={k: os.path.join(RAIZ, v) for k, v in FIGURAS_SITE.items()},
    # o motor é o mesmo do notebook: nenhuma função é reimplementada
    motor=dict(gerar_laudo=gerar_laudo, laudo_em_texto=laudo_em_texto,
               carregar_cinza=carregar_cinza, mascara_encefalo=mascara_encefalo),
)

PORTA = 7860
def _servir():
    aplicacao.run(host="0.0.0.0", port=PORTA, debug=False, use_reloader=False,
                  threaded=True)

if MODO != "teste":
    threading.Thread(target=_servir, daemon=True).start()
    _t.sleep(2.5)
    if EM_COLAB:
        from google.colab.output import eval_js
        print("Abra a interface em:")
        print(eval_js(f"google.colab.kernel.proxyPort({PORTA})"))
    else:
        print(f"Abra a interface em: http://127.0.0.1:{PORTA}")
    print("\nO servidor fica no ar enquanto o ambiente de execução estiver ativo.")
else:
    print("perfil 'teste': aplicação construída, servidor não iniciado de propósito")

crono.marco("interface web no ar")

### 8.4 · O que fazer na interface

1. Abra `/laudo`, escolha um dos exemplos e clique em **Gerar laudo**.
2. Arraste o limiar τ para 0,10 e depois para 0,60 e refaça. Quantos achados aparecem e
   desaparecem? Qual dos dois laudos você assinaria?
3. Vá em `/resultados`: o painel foi montado a partir do `resultados.json` que a seção 9
   vai gravar — os números são desta execução, não de demonstração.
4. Envie uma foto qualquer que não seja uma ressonância. O que o laudo diz? O que ele
   **deveria** dizer? Essa é a pergunta que a seção 6.6 mede.

---
## 9 · Prestação de contas

### 9.1 · O orçamento de 30 minutos

A promessa era executar em até trinta minutos. Abaixo, a medição — não a estimativa.

In [ ]:
display(crono.tabela())
total = sum(e["segundos"] for e in crono.etapas) / 60
print(f"\nTotal medido: {total:.1f} min   |   Orçamento: {cfg.orcamento_total_min:.0f} min   "
      f"|   {'DENTRO' if total <= cfg.orcamento_total_min else 'ACIMA'} do orçamento")
crono.figura("fig/16_orcamento.png"); plt.show()

# Exporta tudo o que o relatório e o README precisam citar, para que nenhum
# número escrito fora do notebook seja digitado à mão.
resultados = {
    "perfil": MODO,
    "config": cfg.dicionario(),
    "dataset": {"particoes": resumo(particoes),
                "sinteticas": len(sinteticas),
                "ood_medical_pills": len(OOD)},
    "treino": {"pasta": PASTA_RUN,
               "epocas_concluidas": int(df_treino["epoch"].max()) if "df_treino" in dir() else None},
    "metricas": {"interno": m_int, "externo": m_ext,
                 "ap_por_classe": {NOMES[c]: curvas_int[c]["ap"] for c in curvas_int},
                 "ece": valor_ece, "tau_operacional": float(TAU),
                 "ponto_interno": ponto_int, "ponto_externo": ponto_ext,
                 "queda_mAP50_dominio_pct": float(queda)},
    "erros": tax, "robustez": tabela_rob, "ood": ood,
    "tempo": {"total_min": float(total),
              "orcamento_min": cfg.orcamento_total_min,
              "etapas": crono.etapas},
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2, default=float)
print("\n➜ resultados.json escrito — é a fonte única dos números do relatório.")

### 9.2 · Rastreabilidade: cada requisito e onde ele foi cumprido

Um entregável só é auditável se cada requisito apontar para a evidência que o satisfaz.

In [ ]:
rastreio = pd.DataFrame([
 ("R1", "Partir do notebook-base e manter o estilo de aula", "todas", "estrutura markdown → código, seção a seção"),
 ("R2", "Usar YOLO26 versão L no mínimo",           "§1, §5", f"cfg.modelo = {cfg.modelo}"),
 ("R3", "Incluir dados sintéticos",                 "§4.1",   f"{len(sinteticas)} imagens com rótulo exato por construção"),
 ("R4", "Incluir um segundo conjunto de dados",     "§3, §3.1", "partição externa por protocolo (175 img) + medical-pills OOD (115 img)"),
 ("R5", "Pipeline com diagnóstico detalhado",       "§6",     "mAP, PR, calibração, taxonomia, domain shift, robustez, OOD"),
 ("R6", "Laudo estruturado por imagem",             "§7",     "JSON versionado com 13 campos por achado"),
 ("R7", "Interface web funcional",                  "§8",     "Flask servindo o template Dtox: 4 páginas + API JSON"),
 ("R8", "Executável em até 30 minutos",             "§5, §9.1", f"{total:.1f} min medidos; teto imposto via time="),
 ("R9", "Projeto inteiro sobre o template Dtox",    "§0.3, §8", "paleta e tipografia do template em 100% das figuras e páginas"),
], columns=["id", "requisito", "seção", "evidência"])
display(rastreio)
rastreio.to_csv("rastreabilidade.csv", index=False)

### 9.3 · O que este projeto **não** prova

A parte mais importante de um relatório técnico é a lista do que ele não sustenta.

1. **Não é diagnóstico.** As classes `negative`/`positive` são convenção de anotação de
   um dataset público, sem laudo clínico verificado por trás. O sistema aprende a
   convenção, não a doença.
2. **A partição externa é um *proxy*.** Geometria de aquisição sugere fonte diferente,
   mas não prova. A conclusão correta é "há deslocamento de distribuição mensurável
   entre esses subconjuntos" — não "isto é validação multicêntrica".
3. **Os dados sintéticos herdam o viés do original.** As lesões coladas vêm do mesmo
   banco de 745 recortes. Eles aumentam variedade de *posição e escala*; não criam
   patologia que o dataset não continha.
4. **Os artefatos são simulados.** Ruído Riciano e ringing de Gibbs reproduzem o modelo
   físico, não a distribuição empírica de um scanner real.
5. **A estabilidade não é probabilidade.** É concordância entre três vistas da mesma
   imagem. Um modelo consistentemente errado exibe estabilidade alta.
6. **O orçamento limita o resultado.** Com 15 minutos de treino, é bem possível que as
   métricas ainda estivessem subindo quando o relógio parou. Se a curva da seção 5.1
   não achatou, o número reportado é um **piso**, não o teto do modelo.
7. **Sem escala física.** Nenhuma medida em milímetros, porque JPEG não carrega
   `PixelSpacing`. O campo existe no laudo e fica vazio até alguém informar a escala.

### 9.4 · Para onde este projeto cresce

- Trocar o teto de tempo por teto de época e comparar o custo real do orçamento.
- Segmentação (`yolo26l-seg.pt`) no lugar de caixa: área de lesão medida em pixels de
  máscara é muito melhor que $w \times h$.
- Calibração *post-hoc* por temperatura sobre as confiâncias, reavaliando o ECE.
- Validação cruzada por protocolo (*leave-one-protocol-out*) em vez de um único holdout.
- Ablação com e sem dados sintéticos, medida na partição externa — a única comparação
  que responde se a síntese ajudou de verdade.

---
## Experimente você mesmo

Na tradição do notebook anterior, cinco experimentos — do mais direto ao mais
interessante:

1. **Mexa no τ.** Rode a seção 7 com `TAU = 0.10` e depois `TAU = 0.60`. Quantas lesões
   aparecem e desaparecem? Qual dos dois laudos você assinaria?

2. **Desligue o sintético.** `cfg.usar_sinteticas = False`, refaça a partir da seção 4.3.
   O mAP **externo** melhora, piora ou não muda? Essa é a única medida que responde se a
   síntese valeu a pena.

3. **Troque o protocolo externo.** Em `particionar`, use `geometria_externa=(256, 256)`.
   O deslocamento fica maior ou menor? Por quê?

4. **Aumente a severidade.** Acrescente uma severidade 4 ao dicionário `ARTEFATOS` e veja
   onde a curva de robustez despenca. Qual artefato quebra o modelo primeiro?

5. **Ataque o controle negativo.** Passe uma foto qualquer — um gato, um documento — pela
   interface web. O que o laudo diz? O que **deveria** dizer? Como você implementaria uma
   verificação de domínio antes da detecção?

---

### Referências

- Redmon, J. et al. *You Only Look Once: Unified, Real-Time Object Detection*. arXiv:1506.02640.
- Ultralytics. *YOLO26 — Unified Real-Time End-to-End Vision Models*. `docs.ultralytics.com/models/yolo26`
- Ultralytics. *Brain Tumor Dataset*. `docs.ultralytics.com/datasets/detect/brain-tumor`
- Ultralytics. *Train Mode — argumento `time`*. `docs.ultralytics.com/modes/train`
- Bolya, D. et al. *TIDE: A General Toolbox for Identifying Object Detection Errors*. ECCV 2020.
- Guo, C. et al. *On Calibration of Modern Neural Networks*. ICML 2017. (ECE)
- Pérez, P. et al. *Poisson Image Editing*. SIGGRAPH 2003. (seamless cloning)
- Gudbjartsson, H.; Patz, S. *The Rician Distribution of Noisy MRI Data*. MRM, 1995.

---

Se você chegou até aqui, parabéns! 🎆 🔥

**Fim!**